# SSL Study 1 — Analysis

## 1  Pre-process

In [ ]:
# pre_process
import re
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import svy
import statsmodels.api as sm
import statsmodels.formula.api as smf
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

import sys
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "study1" else Path.cwd()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
from statsmodels_handler import StatsmodelsHandler
from helpers import make_aesthetic

DATA_FILE = "SSL-Final+(Actual+Run)_May+30,+2026_12.59.csv"
QUOTA_FILE = "yougov_quota.csv"
OUTPUT_DIR = Path("analysis_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# Make the notebook work whether it is launched from study1/ or the project root.
if Path(DATA_FILE).exists():
    DATA_PATH = Path(DATA_FILE)
elif (Path("study1") / DATA_FILE).exists():
    DATA_PATH = Path("study1") / DATA_FILE
else:
    raise FileNotFoundError(f"Could not find {DATA_FILE}")

if Path(QUOTA_FILE).exists():
    QUOTA_PATH = Path(QUOTA_FILE)
elif (Path("study1") / QUOTA_FILE).exists():
    QUOTA_PATH = Path("study1") / QUOTA_FILE
else:
    raise FileNotFoundError(f"Could not find {QUOTA_FILE}")

print(DATA_PATH.resolve())
print(QUOTA_PATH.resolve())

df_raw = pd.read_csv(DATA_PATH, skiprows=[1, 2])
print(df_raw.shape)
display(df_raw.head())

def extract_leading_number(value):
    """
    Pull the first number from responses like '1 (Strongly disagree)' or
    '9 (Very satisfied)'. Keeps plain numbers as they are.
    """
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value)
    text = str(value).strip()
    if text == "":
        return np.nan
    match = re.search(r"^-?\d+(\.\d+)?", text)
    if match:
        return float(match.group())
    return np.nan


def yes_no_ssl(value):
    """Map SSL frequency item to binary ever-use."""
    if pd.isna(value):
        return np.nan
    text = str(value).strip()
    if text == "":
        return np.nan
    return 0 if text == "No - never" else 1


def frequency_code(value):
    mapping = {
        "No - never": 0,
        "Yes - once or twice": 1,
        "Yes - 3-10 times": 2,
        "Yes - 11-20 times": 3,
        "Yes - more than 20 times": 4,
    }
    if pd.isna(value):
        return np.nan
    return mapping.get(str(value).strip(), np.nan)


def standardize(series):
    series = pd.to_numeric(series, errors="coerce")
    sd = series.std(skipna=True)
    if pd.isna(sd) or sd == 0:
        return series * np.nan
    return (series - series.mean(skipna=True)) / sd


def clean_category(series, min_count=10, missing_label="Missing"):
    s = series.astype("object").where(series.notna(), missing_label).astype(str).str.strip()
    counts = s.value_counts(dropna=False)
    rare = counts[counts < min_count].index
    return s.where(~s.isin(rare), "Other")

def recode_sias_text(value):
    mapping = {
        "Not at all characteristic or true of me": 0,
        "Slightly characteristic or true of me": 1,
        "Moderately characteristic or true of me": 2,
        "Very characteristic or true of me": 3,
        "Extremely characteristic or true of me": 4,
    }
    if pd.isna(value):
        return np.nan
    return mapping.get(str(value).strip(), np.nan)


def recode_lsns_text(value):
    mapping = {
        "None": 0,
        "0": 0,
        "1": 1,
        "One": 1,
        "2": 2,
        "Two": 2,
        "3 or 4": 3,
        "3 or 4 people": 3,
        "5 to 8": 4,
        "5 to 8 people": 4,
        "9 or more": 5,
        "9 or more people": 5,
    }
    if pd.isna(value):
        return np.nan
    return mapping.get(str(value).strip(), np.nan)


def recode_tipi_text(value):
    mapping = {
        "Disagree strongly": 1,
        "Disagree moderately": 2,
        "Disagree a little": 3,
        "Neither agree nor disagree": 4,
        "Agree a little": 5,
        "Agree moderately": 6,
        "Agree strongly": 7,
    }
    if pd.isna(value):
        return np.nan
    return mapping.get(str(value).strip(), np.nan)


def reverse_score(series, min_score, max_score):
    return (min_score + max_score) - series


def score_aias4(data):
    items = ["AIAS-4_1", "AIAS-4_2", "AIAS-4_3", "AIAS-4_4"]
    for col in items:
        data[col + "_num"] = data[col].apply(extract_leading_number)
    data["aias4_score"] = data[[col + "_num" for col in items]].mean(axis=1)
    return data


def score_anthrotech(data):
    items = [f"AnthroTech_{i}" for i in range(1, 9)]
    for col in items:
        data[col + "_num"] = data[col].apply(extract_leading_number)
    data["anthrotech_score"] = data[[col + "_num" for col in items]].mean(axis=1)
    return data


def score_sias4(data):
    items = ["SIAS-4_6", "SIAS-4_3", "SIAS-4_8", "SIAS-4_16", "SIAS-4_18", "SIAS-4_19"]
    for col in items:
        data[col + "_num"] = data[col].apply(recode_sias_text)
    data["sias4_score"] = data[[col + "_num" for col in items]].sum(axis=1, min_count=1)
    return data


def score_lsns6(data):
    items = [
        "LSNS-6-Family_1", "LSNS-6-Family_2", "LSNS-6-Family_3",
        "LSNS-6-Friends_1", "LSNS-6-Friends_2", "LSNS-6-Friends_3",
    ]
    for col in items:
        data[col + "_num"] = data[col].apply(recode_lsns_text)
    data["lsns6_score"] = data[[col + "_num" for col in items]].sum(axis=1, min_count=1)
    return data


def score_tipi(data):
    items = [f"TIPI-{i}" for i in range(1, 11)]
    for col in items:
        data[col + "_num"] = data[col].apply(recode_tipi_text)

    data["tipi_extraversion"] = (data["TIPI-1_num"] + reverse_score(data["TIPI-6_num"], 1, 7)) / 2
    data["tipi_agreeableness"] = (data["TIPI-7_num"] + reverse_score(data["TIPI-2_num"], 1, 7)) / 2
    data["tipi_conscientiousness"] = (data["TIPI-3_num"] + reverse_score(data["TIPI-8_num"], 1, 7)) / 2
    data["tipi_emotional_stability"] = (data["TIPI-9_num"] + reverse_score(data["TIPI-4_num"], 1, 7)) / 2
    data["tipi_openness"] = (data["TIPI-5_num"] + reverse_score(data["TIPI-10_num"], 1, 7)) / 2
    return data


def recode_ai_relationship_frequency(value):
    mapping = {
        "Never": 0,
        "Less than a few times a month": 1,
        "A few times a month": 2,
        "Once a week": 3,
        "A few times a week": 4,
        "Once a day": 5,
        "Several times a day": 6,
    }
    if pd.isna(value):
        return np.nan
    return mapping.get(str(value).strip(), np.nan)


def recode_ai_intimacy_frequency(value):
    mapping = {
        "No, never": 0,
        "Yes, once or twice": 1,
        "Yes, occasionally": 2,
        "Yes, frequently": 3,
    }
    if pd.isna(value):
        return np.nan
    return mapping.get(str(value).strip(), np.nan)

def mad_sd(data, scaling_factor=1.4826):
    """
    Compute the Median Absolute Deviation (MAD) of a 1D array with a scaling factor of 1.4826,
    with factor from [1]. This factor is also what Gelman [2] calls the "mad_sd" in Regression and Other Stories (pg 73).

    [1] Leys, Christophe, Christophe Ley, Olivier Klein, Philippe Bernard, and Laurent Licata. “Detecting Outliers: Do Not Use Standard Deviation around the Mean, Use Absolute Deviation around the Median.” Journal of Experimental Social Psychology 49, no. 4 (2013): 764–66. https://doi.org/10.1016/j.jesp.2013.03.013.

    [2] Gelman, Andrew, Jennifer Hill, and Aki Vehtari. Regression and Other Stories. Cambridge University Press, 2020.
    """
    data = np.asarray(data, dtype=float)
    data = data[~np.isnan(data)]
    if len(data) == 0:
        return np.nan
    abs_devs = np.abs(data - np.median(data))
    mad = scaling_factor * np.median(abs_devs)
    return mad


df = df_raw.copy()

# Basic quality/eligibility columns.
df["duration_seconds"] = pd.to_numeric(df["Duration (in seconds)"], errors="coerce")
df["finished_bool"] = df["Finished"].astype(str).str.lower().eq("true")
df["progress_num"] = pd.to_numeric(df["Progress"], errors="coerce")
df["committed"] = df["commit"].eq("Yes, I will")
df["eligible_chatbot_user"] = df["use_chatbots"].eq("Yes")

# SSL domain indicators.
df["used_moral_ssl"] = df["domain_e_freq"].apply(yes_no_ssl)
df["used_personal_ssl"] = df["domain_p_freq"].apply(yes_no_ssl)
df["used_conventional_ssl"] = df["domain_c_freq"].apply(yes_no_ssl)
ssl_cols = ["used_personal_ssl", "used_conventional_ssl", "used_moral_ssl"]
df["n_ssl_domains"] = df[ssl_cols].sum(axis=1, min_count=1)
df["used_any_ssl"] = np.where(df[ssl_cols].isna().all(axis=1), np.nan, (df[ssl_cols].eq(1).any(axis=1)).astype(int))

# Frequency-coded domain outcomes.
df["moral_ssl_freq_code"] = df["domain_e_freq"].apply(frequency_code)
df["personal_ssl_freq_code"] = df["domain_p_freq"].apply(frequency_code)
df["conventional_ssl_freq_code"] = df["domain_c_freq"].apply(frequency_code)

speed_rows = []
for n_domains, group in df.groupby("n_ssl_domains", dropna=False):
    durations = group["duration_seconds"].dropna()
    median_duration = durations.median()
    group_mad_sd = mad_sd(durations)
    threshold = median_duration - 2 * group_mad_sd if pd.notna(group_mad_sd) else np.nan
    idx = group.index
    df.loc[idx, "duration_group_median"] = median_duration
    df.loc[idx, "duration_group_mad_sd"] = group_mad_sd
    df.loc[idx, "went_fast_threshold"] = threshold
    df.loc[idx, "went_fast"] = df.loc[idx, "duration_seconds"] < threshold
    speed_rows.append({
        "n_ssl_domains": n_domains,
        "n": len(group),
        "median_duration": median_duration,
        "mad_sd": group_mad_sd,
        "fast_threshold": threshold,
        "n_went_fast": int(df.loc[idx, "went_fast"].sum()),
    })

speed_table = pd.DataFrame(speed_rows).sort_values("n_ssl_domains")
display(speed_table)

df = score_aias4(df)
df = score_anthrotech(df)
df = score_sias4(df)
df = score_lsns6(df)
df = score_tipi(df)

for col in ["air_friend", "air_rship"]:
    df[col + "_num"] = df[col].apply(recode_ai_relationship_frequency)

for col in ["ais_chosen", "ais_shared", "ais_embarass", "ais_grief"]:
    df[col + "_num"] = df[col].apply(recode_ai_intimacy_frequency)

def recode(series, from_min, from_max, to_min, to_max):
    """Linearly rescale a numeric series from [from_min, from_max] to [to_min, to_max]."""
    return to_min + (series - from_min) * (to_max - to_min) / (from_max - from_min)


# air_* items are 0–6; ais_* items are 0–3 — rescale ais_* to 0–6 before averaging
_AIS_COLS = ["ais_chosen_num", "ais_shared_num", "ais_embarass_num", "ais_grief_num"]
for _col in _AIS_COLS:
    df[_col] = recode(df[_col], from_min=0, from_max=3, to_min=0, to_max=6)

social_idx_vars = [
    "air_friend_num", "air_rship_num",
    "ais_chosen_num", "ais_shared_num", "ais_embarass_num", "ais_grief_num",
]
df["ai_social_use_index"] = df[social_idx_vars].mean(axis=1)

df["use_freq_code"] = df["use_freq"].map({
    "Rarely": 1,
    "Sometimes": 2,
    "Frequently": 3,
    "Every day": 4,
})

df["age_num"] = pd.to_numeric(df["age"], errors="coerce")
df["ideo_num"] = df["ideo5"].map({
    "Extremely liberal": 1,
    "Liberal": 2,
    "Slightly liberal": 3,
    "Moderate; middle of the road": 4,
    "Slightly conservative": 5,
    "Conservative": 6,
    "Extremely conservative": 7,
})

# Collapsed demographic factors to keep regression degrees of freedom sane.

# Low-dimensional demographic indicators for logistic models.
income_order = {
    "Less than $25,000": 1,
    "$25,000-$49,999": 2,
    "$50,000-$74,999": 3,
    "$75,000-$99,999": 4,
    "$100,000-$149,999": 5,
    "$150,000 or more": 6,
}
df["income_ord"] = df["income"].map(income_order)
df["college_degree"] = df["education"].isin([
    "Bachelor’s degree",
    "Graduate or professional degree (MA, MS, MBA, PhD, JD, MD, DDS etc.)",
]).astype(float)
df["gender_male"] = df["gender"].eq("Male").astype(float)
df["gender_other"] = (~df["gender"].isin(["Male", "Female"])).astype(float)
df["race_white"] = df["race"].eq("White or Caucasian").astype(float)
df["race_black"] = df["race"].eq("Black or African American").astype(float)
df["pid_republican"] = df["pid3"].eq("Republican").astype(float)
df["pid_democrat"] = df["pid3"].eq("Democrat").astype(float)
df["employed"] = df["employment"].astype(str).str.contains("Employed|Self-employed", regex=True).astype(float)
df["retired"] = df["employment"].eq("Retired").astype(float)

df["gender_c"] = clean_category(df["gender"], min_count=15)
df["pid3_c"] = clean_category(df["pid3"], min_count=15)
df["income_c"] = clean_category(df["income"], min_count=15)
df["education_c"] = clean_category(df["education"], min_count=15)
df["employment_c"] = clean_category(df["employment"], min_count=20)

def collapse_race(value):
    if pd.isna(value):
        return "Missing"
    text = str(value)
    if "," in text:
        return "Multiracial"
    if text in ["White or Caucasian", "Black or African American", "Asian"]:
        return text
    return "Other"

df["race_c"] = df["race"].apply(collapse_race)

scale_score_cols = [
    "aias4_score", "anthrotech_score", "sias4_score", "lsns6_score",
    "tipi_extraversion", "tipi_agreeableness", "tipi_conscientiousness",
    "tipi_emotional_stability", "tipi_openness", "ai_social_use_index",
    "use_freq_code", "age_num", "ideo_num", "income_ord",
]

for col in scale_score_cols:
    df[col + "_z"] = standardize(df[col])

display(df[scale_score_cols].describe().T)

EXCLUDE_WENT_FAST = True

base_mask = (
    df["finished_bool"]
    & df["progress_num"].ge(100)
    & df["committed"]
    & df["eligible_chatbot_user"]
)

analysis_mask = base_mask.copy()
if EXCLUDE_WENT_FAST:
    analysis_mask = analysis_mask & ~df["went_fast"].fillna(False)

df_analysis = df.loc[analysis_mask].copy()

sample_summary = pd.DataFrame({
    "sample": ["raw", "base_eligible", "analysis"],
    "n": [len(df), int(base_mask.sum()), len(df_analysis)],
    "went_fast_n": [int(df["went_fast"].sum()), int(df.loc[base_mask, "went_fast"].sum()), int(df_analysis["went_fast"].sum())],
})
display(sample_summary)

df.to_csv(OUTPUT_DIR / "study1_qualtrics_scored_with_flags.csv", index=False)
df_analysis.to_csv(OUTPUT_DIR / "study1_analysis_sample.csv", index=False)
print(OUTPUT_DIR.resolve())


In [ ]:
# viz_constants
# Shared visualization constants — defined once, used across all sections.
import textwrap
from helpers import make_aesthetic
import matplotlib.lines as mlines
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams.update({
    "font.family":       "sans-serif",
    "font.sans-serif":   ["Helvetica", "Arial", "DejaVu Sans"],
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "pdf.fonttype":      42,
    "ps.fonttype":       42,
})

####################
# Font hierarchy
####################

BASE = 11
FONT = {
    "small":  BASE,          # 11 — y-axis labels, tick labels, legend entries
    "medium": BASE * 1.25,   # ~14 — axis labels, bucket headers, legend title
    "large":  BASE * 1.75,   # ~19 — plot title
}

COLORS = make_aesthetic(font_scale=1.2, bold_title=True)

DOMAIN_ORDER_RAW = ["personal_psychological", "societal_conventional", "moral"]
DOMAIN_DISPLAY   = {"personal_psychological": "Personal",
                    "societal_conventional":  "Conventional",
                    "moral":                  "Moral"}
DOMAIN_MARKERS   = {"personal_psychological": "o",
                    "societal_conventional":  "D",
                    "moral":                  "^"}
# Domain colors from make_aesthetic default palette (indices 0, 4, 7)
# Bucket colors occupy indices 5, 6, 8, 12 — no conflict
DOMAIN_COLORS    = {"personal_psychological": COLORS[0],   # #00A896 teal-green
                    "societal_conventional":  COLORS[7],   # #020887 dark-blue
                    "moral":                  COLORS[4]}   # #F45B69 coral-red
DOMAIN_SIZES     = {"personal_psychological": 7,
                    "societal_conventional":  6,
                    "moral":                  7}

BUCKET_ORDER   = ["practical", "productive", "social", "entertain"]
BUCKET_DISPLAY = {"practical":  "Practical",
                  "productive": "Productivity",
                  "social":     "Social",
                  "entertain":  "Entertainment"}
BUCKET_COLORS  = dict(zip(BUCKET_ORDER, [COLORS[5], COLORS[6], COLORS[8], COLORS[12]]))


## 2  Weighting

Pre-weighting diagnostic → rake LLM freq (high/low) × age (18-44/45+) × income → cap at 5 → trim at p95.

In [ ]:
# pre_weighting_diagnostic
# Goal: identify which demographic variables (1) predict any SSL and
#       (2) have |sample − YouGov| > 5pp.  Those should be raked on.

# ── 1. Unweighted logistic regression: predictors of any SSL ───────────────
_pre_preds = [
    "age_num_z", "gender_male", "gender_other",
    "race_white", "race_black",
    "pid_republican", "pid_democrat",
    "income_ord_z",
    "use_freq_code_z",          # LLM usage frequency
]
_model_df = df_analysis[["used_any_ssl"] + _pre_preds].dropna().copy()
_y = _model_df["used_any_ssl"].astype(int)
_X = sm.add_constant(_model_df[_pre_preds], has_constant="add")

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    _logit = sm.Logit(_y, _X).fit(disp=False, maxiter=200)

pred_table_preweight = (
    pd.DataFrame({
        "term": _logit.params.index,
        "OR":   np.exp(_logit.params.values).round(2),
        "p":    _logit.pvalues.values.round(3),
    })
    .query("term != 'const'")
    .sort_values("p")
    .assign(sig=lambda d: d["p"].apply(
        lambda p: "***" if p < .001 else ("**" if p < .01 else ("*" if p < .05 else ""))))
)
print(f"n = {int(_logit.nobs)},  pseudo-R² = {_logit.prsquared:.3f}\n")
display(pred_table_preweight)


# ── 2. Sample vs. YouGov balance check (re-creates groupings locally) ──────
def _age_grp(a):
    a = pd.to_numeric(a, errors="coerce")
    if pd.isna(a):  return None
    if a <= 29:     return "18-29"
    if a <= 44:     return "30-44"
    if a <= 64:     return "45-64"
    return "65+"

def _inc_grp(v):
    if pd.isna(v):  return None
    s = str(v).strip()
    if s in ["Less than $25,000", "$25,000-$49,999", "Under $50K"]:      return "<$50K"
    if s in ["$50,000-$74,999", "$75,000-$99,999", "$50-100K"]:          return "$50-100K"
    if s in ["$100,000-$149,999", "$150,000 or more", "$100K or more"]:  return "$100K+"
    return None

_qdf = pd.read_csv(QUOTA_PATH)
_yg  = {(r["category"], r["group"]): float(r["yes"]) for _, r in _qdf.iterrows()}

_age_s = df_analysis["age"].apply(_age_grp)
_inc_s = df_analysis["income"].apply(_inc_grp)

# (category, level, mask, yougov_pct or None)
# None = no external target available — sample % shown descriptively
_LLM_TARGETS = {"Every day": 11.0, "Frequently": 22.0, "Sometimes": 34.0, "Rarely": 33.0}

_checks = [
    ("Gender",        "Male",        df_analysis["gender"].eq("Male"),                              _yg.get(("Gender",   "Male"))),
    ("Gender",        "Female",      df_analysis["gender"].eq("Female"),                            _yg.get(("Gender",   "Female"))),
    ("Age",           "18-29",       _age_s.eq("18-29"),                                            _yg.get(("Age",      "18-29"))),
    ("Age",           "30-44",       _age_s.eq("30-44"),                                            _yg.get(("Age",      "30-44"))),
    ("Age",           "45-64",       _age_s.eq("45-64"),                                            _yg.get(("Age",      "45-64"))),
    ("Age",           "65+",         _age_s.eq("65+"),                                              _yg.get(("Age",      "65+"))),
    ("Income",        "<$50K",       _inc_s.eq("<$50K"),                                            _yg.get(("Income",   "<$50K"))),
    ("Income",        "$50-100K",    _inc_s.eq("$50-100K"),                                         _yg.get(("Income",   "$50-100K"))),
    ("Income",        "$100K+",      _inc_s.eq("$100K+"),                                           _yg.get(("Income",   "$100K+"))),
    ("Party ID",      "Democrat",    df_analysis["pid3"].eq("Democrat"),                            _yg.get(("Party ID", "Democrat"))),
    ("Party ID",      "Republican",  df_analysis["pid3"].eq("Republican"),                          _yg.get(("Party ID", "Republican"))),
    ("Race",          "White",       df_analysis["race"].eq("White or Caucasian"),                  _yg.get(("Race",     "White"))),
    ("Race",          "Black",       df_analysis["race"].eq("Black or African American"),           _yg.get(("Race",     "Black"))),
    # LLM frequency — using our own pop-est targets
    ("LLM Frequency", "Every day",   df_analysis["use_freq"].eq("Every day"),                      _LLM_TARGETS["Every day"]),
    ("LLM Frequency", "Frequently",  df_analysis["use_freq"].eq("Frequently"),                     _LLM_TARGETS["Frequently"]),
    ("LLM Frequency", "Sometimes",   df_analysis["use_freq"].eq("Sometimes"),                      _LLM_TARGETS["Sometimes"]),
    ("LLM Frequency", "Rarely",      df_analysis["use_freq"].eq("Rarely"),                         _LLM_TARGETS["Rarely"]),
]

balance_preweight = []
for cat, grp, mask, yg in _checks:
    s = round(mask.mean() * 100, 1)
    d = round(s - yg, 1) if yg is not None else np.nan
    balance_preweight.append({
        "variable":   cat,
        "level":      grp,
        "yougov":     yg if yg is not None else np.nan,
        "sample":     s,
        "delta":      d,
        "|delta|>5":  (abs(d) > 5) if not np.isnan(d) else None,
    })
balance_preweight_df = pd.DataFrame(balance_preweight)
display(balance_preweight_df)


# ── 3. Recommendation ─────────────────────────────────────────────────────
_TERM_TO_CAT = {
    "age_num_z":      "Age",           "gender_male":    "Gender",
    "gender_other":   "Gender",        "race_white":     "Race",
    "race_black":     "Race",          "pid_democrat":   "Party ID",
    "pid_republican": "Party ID",      "income_ord_z":   "Income",
    "use_freq_code_z":"LLM Frequency",
}
sig_cats   = {_TERM_TO_CAT[t] for t in pred_table_preweight.query("p < .05")["term"]
              if _TERM_TO_CAT.get(t)}
delta_cats = {r["variable"] for _, r in balance_preweight_df.iterrows() if abs(r["delta"]) > 5}
recommend  = sig_cats & delta_cats

print("\nSignificant demographic predictors (p < .05):", sig_cats)
print("Variables with |delta| > 5pp from YouGov:    ", delta_cats)
print("→ Recommended for raking:                    ", recommend)


In [ ]:
# weighting
def age_quota_group(age):
    age = pd.to_numeric(age, errors="coerce")
    if pd.isna(age):
        return "Other"
    if 18 <= age <= 29:
        return "18-29"
    if 30 <= age <= 44:
        return "30-44"
    if 45 <= age <= 64:
        return "45-64"
    if age >= 65:
        return "65+"
    return "Other"


def income_quota_group(value):
    if pd.isna(value):
        return "Other"
    text = str(value).strip()
    if text in ["Less than $25,000", "$25,000-$49,999", "Under $50K"]:
        return "<$50K"
    if text in ["$50,000-$74,999", "$75,000-$99,999", "$50-100K"]:
        return "$50-100K"
    if text in ["$100,000-$149,999", "$150,000 or more", "$100K or more"]:
        return "$100K+"
    return "Other"


def llm_freq_quota_group(value):
    if pd.isna(value):
        return "Other"
    text = str(value).strip()
    if text in ["Every day", "Frequently", "Sometimes", "Rarely"]:
        return text
    return "Other"


def normalize_props(values):
    total = sum(values.values())
    return {k: v / total for k, v in values.items() if total > 0}


def quota_targets(category, quota_df, value_col="yes"):
    sub = quota_df[quota_df["category"].eq(category)].copy()
    out = sub.groupby("group", dropna=False)[value_col].sum().to_dict()
    return normalize_props(out)


def complete_targets(data, var, target_props):
    present = data[var].value_counts(normalize=True, dropna=False).to_dict()
    present = {str(k): float(v) for k, v in present.items()}
    target_props = {str(k): float(v) for k, v in target_props.items()}

    unsupported = sorted([k for k in target_props if k not in present])
    sample_only = {k: v for k, v in present.items() if k not in target_props}
    sample_only_total = sum(sample_only.values())
    known_present = {k: v for k, v in target_props.items() if k in present}
    known_total = sum(known_present.values())
    if known_total > 0:
        known_present = {k: v / known_total * (1 - sample_only_total) for k, v in known_present.items()}
    completed = {**known_present, **sample_only}
    completed_total = sum(completed.values())
    completed = {k: v / completed_total for k, v in completed.items()}
    return completed, unsupported


def one_margin_adjust(weights, domain, target_props):
    domain = pd.Series(domain).astype(str)
    weights = np.asarray(weights, dtype=float).copy()
    total_weight = weights.sum()
    for level, prop in target_props.items():
        idx = domain.eq(str(level)).to_numpy()
        current = weights[idx].sum()
        target = prop * total_weight
        if current > 0:
            weights[idx] *= target / current
    return weights


def rake_to_targets(data, controls, max_iter=500, tol=1e-8):
    weights = np.ones(len(data), dtype=float)
    diagnostics = []
    for iteration in range(1, max_iter + 1):
        previous = weights.copy()
        for var, target_props in controls.items():
            weights = one_margin_adjust(weights, data[var], target_props)
        weights *= len(weights) / weights.sum()
        max_change = np.max(np.abs(weights - previous))
        diagnostics.append({"iteration": iteration, "max_abs_weight_change": max_change})
        if max_change < tol:
            break
    return weights, pd.DataFrame(diagnostics)


def weighted_mean(series, weights):
    valid = series.notna() & pd.Series(weights, index=series.index).notna()
    if valid.sum() == 0:
        return np.nan
    return np.average(series[valid].astype(float), weights=pd.Series(weights, index=series.index)[valid])


quota_df = pd.read_csv(QUOTA_PATH)

for target_df in [df, df_analysis]:
    target_df["quota_age"] = target_df["age"].apply(age_quota_group)
    target_df["quota_income"] = target_df["income"].apply(income_quota_group)
    target_df["quota_llm_freq"] = target_df["use_freq"].apply(llm_freq_quota_group)

llm_freq_overall = normalize_props({
    "Every day": 11,
    "Frequently": 22,
    "Sometimes": 34,
    "Rarely": 33,
})

raw_controls = {
    "quota_llm_freq": llm_freq_overall,
    "quota_age": quota_targets("Age", quota_df),
    "quota_income": quota_targets("Income", quota_df),
}

controls = {}
unsupported_rows = []
for var, target in raw_controls.items():
    controls[var], unsupported = complete_targets(df_analysis, var, target)
    unsupported_rows.extend({"variable": var, "unsupported_target_cell": cell} for cell in unsupported)

unsupported_targets = pd.DataFrame(unsupported_rows)
df_analysis["yougov_weight"], rake_diagnostics = rake_to_targets(df_analysis, controls)

weight_summary = pd.DataFrame({
    "n": [len(df_analysis)],
    "sum_weights": [df_analysis["yougov_weight"].sum()],
    "mean_weight": [df_analysis["yougov_weight"].mean()],
    "min_weight": [df_analysis["yougov_weight"].min()],
    "max_weight": [df_analysis["yougov_weight"].max()],
    "weight_cv": [df_analysis["yougov_weight"].std() / df_analysis["yougov_weight"].mean()],
    "kish_effective_n": [(df_analysis["yougov_weight"].sum() ** 2) / (df_analysis["yougov_weight"] ** 2).sum()],
    "iterations": [len(rake_diagnostics)],
    "backend": ["local_ipf"],
})

display(weight_summary)
display(rake_diagnostics.tail())
if not unsupported_targets.empty:
    print("Unsupported target cells dropped before raking:")
    display(unsupported_targets)

balance_rows = []
for var, target in controls.items():
    for level, target_prop in target.items():
        idx = df_analysis[var].astype(str).eq(str(level))
        balance_rows.append({
            "variable": var,
            "level": level,
            "target_percent": target_prop * 100,
            "unweighted_percent": idx.mean() * 100,
            "weighted_percent": df_analysis.loc[idx, "yougov_weight"].sum() / df_analysis["yougov_weight"].sum() * 100,
        })
weight_balance = pd.DataFrame(balance_rows)
display(weight_balance)

weight_summary.to_csv(OUTPUT_DIR / "yougov_weight_summary.csv", index=False)
rake_diagnostics.to_csv(OUTPUT_DIR / "yougov_rake_diagnostics.csv", index=False)
weight_balance.to_csv(OUTPUT_DIR / "yougov_weight_balance.csv", index=False)
unsupported_targets.to_csv(OUTPUT_DIR / "yougov_unsupported_target_cells.csv", index=False)


In [ ]:
# analysis_weighting
# Rake on 3 collapsed controls:
#   LLM frequency 2-group (High=daily+frequent / Low=sometimes+rarely)
#   Age 2-group (18-44 / 45+)
#   Income 3-group (unchanged from YouGov)

def normalize_weight(w, n=None):
    w = np.asarray(w, dtype=float)
    if n is None:
        n = len(w)
    return w * n / np.nansum(w)


def cap_and_redistribute_weight(w, cap=5.0, tol=1e-10, max_iter=1000):
    """Cap weights while preserving sum(weights) == n."""
    w = normalize_weight(w)
    n = len(w)
    capped = np.zeros(n, dtype=bool)
    for _ in range(max_iter):
        over = w > cap
        if not over.any():
            break
        capped = capped | over
        w[capped] = cap
        remaining = ~capped
        remaining_total = n - w[capped].sum()
        if remaining.any() and remaining_total > 0:
            w[remaining] = w[remaining] * remaining_total / w[remaining].sum()
        if np.nanmax(w[~capped]) <= cap + tol:
            break
    return w


# ── Collapsed grouping variables ──────────────────────────────────────────
def age2_quota_group(age):
    age = pd.to_numeric(age, errors="coerce")
    if pd.isna(age):    return "Other"
    if 18 <= age <= 44: return "18-44"
    if age >= 45:       return "45+"
    return "Other"


def llm_freq2_quota_group(value):
    if pd.isna(value): return "Other"
    text = str(value).strip()
    if text in ["Every day", "Frequently"]: return "High"
    if text in ["Sometimes", "Rarely"]:     return "Low"
    return "Other"


for target_df in [df, df_analysis]:
    target_df["quota_age2"]      = target_df["age"].apply(age2_quota_group)
    target_df["quota_llm_freq2"] = target_df["use_freq"].apply(llm_freq2_quota_group)


# ── Population targets ────────────────────────────────────────────────────
# LLM freq 2-group: 11+22=33 high, 34+33=67 low (from our pop-est)
llm_freq2_targets = normalize_props({"High": 33, "Low": 67})

# Age 2-group: sum YouGov LLM-user (yes) proportions across collapsed bins
_age_yes = {r["group"]: float(r["yes"])
            for _, r in quota_df[quota_df["category"] == "Age"].iterrows()}
age2_targets = normalize_props({
    "18-44": _age_yes.get("18-29", 0) + _age_yes.get("30-44", 0),
    "45+":   _age_yes.get("45-64", 0) + _age_yes.get("65+",  0),
})

# Income 3-group: direct from YouGov LLM-user yes column
_inc_yes = {r["group"]: float(r["yes"])
            for _, r in quota_df[quota_df["category"] == "Income"].iterrows()}
income_targets = normalize_props(_inc_yes)

# Apply complete_targets to handle any levels present in sample but not in targets
raw_controls3 = {
    "quota_llm_freq2": llm_freq2_targets,
    "quota_age2":      age2_targets,
    "quota_income":    income_targets,
}
controls3 = {}
for var, tgt in raw_controls3.items():
    controls3[var], _ = complete_targets(df_analysis, var, tgt)


# ── Rake → cap → trim ───────────────────────────────────────────────────
df_analysis["weight_raw"], rake_diagnostics = rake_to_targets(df_analysis, controls3)

# Hard cap at 5 to redistribute the most extreme weights
df_analysis["weight_cap5"] = cap_and_redistribute_weight(
    df_analysis["weight_raw"].to_numpy(), cap=5.0)

# Percentile trim at 95th to further reduce variance, then renormalize
def trim_weights(w, pct=95):
    w = np.asarray(w, dtype=float)
    ceiling = np.nanpercentile(w, pct)
    w = np.minimum(w, ceiling)
    return w * len(w) / w.sum()

df_analysis["weight_trimmed"] = trim_weights(df_analysis["weight_cap5"].to_numpy())
REPORT_WEIGHT_COL = "weight_trimmed"

weight_summary = pd.DataFrame({
    "n":                [len(df_analysis)],
    "sum_weights":      [df_analysis[REPORT_WEIGHT_COL].sum()],
    "mean_weight":      [df_analysis[REPORT_WEIGHT_COL].mean()],
    "min_weight":       [df_analysis[REPORT_WEIGHT_COL].min()],
    "max_weight":       [df_analysis[REPORT_WEIGHT_COL].max()],
    "weight_cv":        [df_analysis[REPORT_WEIGHT_COL].std() / df_analysis[REPORT_WEIGHT_COL].mean()],
    "kish_effective_n": [(df_analysis[REPORT_WEIGHT_COL].sum() ** 2) /
                         (df_analysis[REPORT_WEIGHT_COL] ** 2).sum()],
    "iterations":       [len(rake_diagnostics)],
    "scheme":           ["freq2_age2_income_cap5_trim95"],
})
display(weight_summary)
display(rake_diagnostics.tail())


# ── Comprehensive balance table ───────────────────────────────────────────
_qdf    = pd.read_csv(QUOTA_PATH)
_yg_yes = {(r["category"], r["group"]): float(r["yes"]) for _, r in _qdf.iterrows()}


def _add_balance(rows, variable, mask, yougov_pct, level):
    sample_pct   = round(mask.mean() * 100, 1)
    weighted_pct = round(
        df_analysis.loc[mask, REPORT_WEIGHT_COL].sum() /
        df_analysis[REPORT_WEIGHT_COL].sum() * 100, 1)
    rows.append({
        "variable":       variable,
        "level":          level,
        "yougov":         yougov_pct,
        "sample":         sample_pct,
        "weighted":       weighted_pct,
        "delta_sample":   round(sample_pct   - yougov_pct, 1),
        "delta_weighted": round(weighted_pct - yougov_pct, 1),
    })


balance_rows = []

# Raked variables — delta_weighted should be ~0
for grp, prop in llm_freq2_targets.items():
    _add_balance(balance_rows, "LLM Frequency (raked)",
                 df_analysis["quota_llm_freq2"].eq(grp), round(prop * 100, 1), grp)

for grp, prop in age2_targets.items():
    _add_balance(balance_rows, "Age (raked)",
                 df_analysis["quota_age2"].eq(grp), round(prop * 100, 1), grp)

for grp in ["<$50K", "$50-100K", "$100K+"]:
    t = _yg_yes.get(("Income", grp))
    if t: _add_balance(balance_rows, "Income (raked)",
                       df_analysis["quota_income"].eq(grp), t, grp)

# Non-raked variables — shows residual imbalance
for grp in ["Male", "Female"]:
    t = _yg_yes.get(("Gender", grp))
    if t: _add_balance(balance_rows, "Gender",
                       df_analysis["gender"].eq(grp), t, grp)

for grp in ["Democrat", "Republican", "Independent"]:
    t = _yg_yes.get(("Party ID", grp))
    if t: _add_balance(balance_rows, "Party ID",
                       df_analysis["pid3"].eq(grp), t, grp)

for grp, survey_val in {"White": "White or Caucasian",
                         "Black": "Black or African American"}.items():
    t = _yg_yes.get(("Race", grp))
    if t: _add_balance(balance_rows, "Race",
                       df_analysis["race"].eq(survey_val), t, grp)

weight_balance = pd.DataFrame(balance_rows)
display(weight_balance)

weight_summary.to_csv(OUTPUT_DIR / "weight_summary.csv", index=False)
rake_diagnostics.to_csv(OUTPUT_DIR / "weight_rake_diagnostics.csv", index=False)
weight_balance.to_csv(OUTPUT_DIR / "weight_balance.csv", index=False)


def make_svy_sample(data, weight_col=REPORT_WEIGHT_COL):
    return svy.Sample(pl.from_pandas(data.reset_index(drop=True)), design=svy.Design(wgt=weight_col))


def svy_mean_row(data, y, weight_col=REPORT_WEIGHT_COL):
    sample = make_svy_sample(data[[y, weight_col]].dropna(), weight_col)
    row = sample.estimation.mean(y).to_dicts()[0]
    return {"estimate": row["est"], "se": row["se"], "ci_low": row["lci"], "ci_high": row["uci"]}


def svy_prop_level_row(data, y, level=1, weight_col=REPORT_WEIGHT_COL):
    sample_data = data[[y, weight_col]].dropna().copy()
    sample_data[y] = sample_data[y].astype(int)
    sample = make_svy_sample(sample_data, weight_col)
    rows = sample.estimation.prop(y).to_dicts()
    for row in rows:
        if str(row.get("y_level")) == str(level):
            return {"estimate": row["est"], "se": row["se"], "ci_low": row["lci"], "ci_high": row["uci"]}
    return {"estimate": np.nan, "se": np.nan, "ci_low": np.nan, "ci_high": np.nan}


def svy_tabulate_df(data, rowvar, colvar=None, weight_col=REPORT_WEIGHT_COL, units="percent"):
    sample_cols = [rowvar, weight_col] + ([colvar] if colvar else [])
    sample_data = data[sample_cols].dropna().copy()
    sample = make_svy_sample(sample_data, weight_col)
    tab = sample.categorical.tabulate(rowvar=rowvar, colvar=colvar, units=units, drop_nulls=True)
    out = tab.to_polars().to_pandas().rename(columns={
        rowvar: "row",
        "est": f"svy_{units}",
        "se": f"svy_{units}_se",
        "lci": f"svy_{units}_ci_low",
        "uci": f"svy_{units}_ci_high",
    })
    if colvar:
        out = out.rename(columns={colvar: "col"})
    return out


## 3  Prevalence

In [ ]:
# analysis_ssl_prevalence_and_frequencies

domain_map = {
    "any_ssl": "used_any_ssl",
    "personal_psychological": "used_personal_ssl",
    "societal_conventional": "used_conventional_ssl",
    "moral": "used_moral_ssl",
}

prevalence_rows = []
for label, col in domain_map.items():
    valid = df_analysis[col].dropna()
    svy_est = svy_prop_level_row(df_analysis.loc[valid.index], col, level=1, weight_col=REPORT_WEIGHT_COL)
    prevalence_rows.append({
        "outcome": label,
        "weight": REPORT_WEIGHT_COL,
        "n_valid": len(valid),
        "n_used": int(valid.sum()),
        "proportion": valid.mean(),
        "percent": valid.mean() * 100,
        "weighted_proportion": svy_est["estimate"],
        "weighted_percent": svy_est["estimate"] * 100,
        "weighted_ci_low": svy_est["ci_low"],
        "weighted_ci_high": svy_est["ci_high"],
        "weighted_ci_low_percent": svy_est["ci_low"] * 100,
        "weighted_ci_high_percent": svy_est["ci_high"] * 100,
        "weighted_se": svy_est["se"],
    })

prevalence_table = pd.DataFrame(prevalence_rows)
for col in ["percent", "weighted_percent", "weighted_ci_low_percent", "weighted_ci_high_percent"]:
    prevalence_table[col] = prevalence_table[col].round(1)
display(prevalence_table)
prevalence_table.to_csv(OUTPUT_DIR / "ssl_prevalence.csv", index=False)

frequency_rows = []
for label, col in {
    "personal_psychological": "domain_p_freq",
    "societal_conventional": "domain_c_freq",
    "moral": "domain_e_freq",
}.items():
    counts = df_analysis[col].value_counts(dropna=False)
    denom = counts.sum()
    weighted_denom = df_analysis[REPORT_WEIGHT_COL].sum()
    tab = svy_tabulate_df(df_analysis, rowvar=col, weight_col=REPORT_WEIGHT_COL, units="percent")
    tab = tab.rename(columns={"row": "response"})
    for response, n in counts.items():
        idx = df_analysis[col].eq(response) if pd.notna(response) else df_analysis[col].isna()
        tab_row = tab[tab["response"].astype(str).eq(str(response))]
        svy_values = tab_row.iloc[0].to_dict() if len(tab_row) else {}
        frequency_rows.append({
            "domain": label,
            "response": response,
            "weight": REPORT_WEIGHT_COL,
            "n": int(n),
            "percent": round(n / denom * 100, 1),
            "weighted_n": df_analysis.loc[idx, REPORT_WEIGHT_COL].sum(),
            "weighted_percent": round(df_analysis.loc[idx, REPORT_WEIGHT_COL].sum() / weighted_denom * 100, 1),
            "weighted_percent_se": svy_values.get("svy_percent_se", np.nan),
            "weighted_ci_low_percent": svy_values.get("svy_percent_ci_low", np.nan),
            "weighted_ci_high_percent": svy_values.get("svy_percent_ci_high", np.nan),
        })

frequency_table = pd.DataFrame(frequency_rows)
display(frequency_table)
frequency_table.to_csv(OUTPUT_DIR / "ssl_domain_frequency_distributions.csv", index=False)

any_frequency = df_analysis["n_ssl_domains"].value_counts(dropna=False).sort_index().rename_axis("n_ssl_domains").reset_index(name="n")
any_frequency["percent"] = (any_frequency["n"] / any_frequency["n"].sum() * 100).round(1)
weighted_any = df_analysis.groupby("n_ssl_domains", dropna=False)[REPORT_WEIGHT_COL].sum().rename("weighted_n").reset_index()
any_frequency = any_frequency.merge(weighted_any, on="n_ssl_domains", how="left")
any_frequency["weight"] = REPORT_WEIGHT_COL
any_frequency["weighted_percent"] = (any_frequency["weighted_n"] / any_frequency["weighted_n"].sum() * 100).round(1)
any_tab = svy_tabulate_df(df_analysis.assign(n_ssl_domains_str=df_analysis["n_ssl_domains"].astype(str)), rowvar="n_ssl_domains_str", weight_col=REPORT_WEIGHT_COL, units="percent")
any_tab["n_ssl_domains"] = pd.to_numeric(any_tab["row"], errors="coerce")
any_frequency = any_frequency.merge(
    any_tab[["n_ssl_domains", "svy_percent_se", "svy_percent_ci_low", "svy_percent_ci_high"]],
    on="n_ssl_domains",
    how="left",
)
any_frequency = any_frequency.rename(columns={
    "svy_percent_se": "weighted_percent_se",
    "svy_percent_ci_low": "weighted_ci_low_percent",
    "svy_percent_ci_high": "weighted_ci_high_percent",
})
display(any_frequency)
any_frequency.to_csv(OUTPUT_DIR / "ssl_number_of_domains_distribution.csv", index=False)

In [ ]:
frequency_table

In [ ]:
####################
# Install CmdStanPy
####################

!pip -q install cmdstanpy
!python -m cmdstanpy.install_cmdstan --quiet


####################
# Imports
####################

import numpy as np
import pandas as pd
import cmdstanpy as csp


####################
# Data
####################

N_PER_ESTIMATE = 800

p1_mean = np.array([0.56, 0.50, 0.45])
p2_mean = np.array([0.20, 0.24, 0.18, 0.22])


####################
# Helpers
####################

def logit(p):
    """Convert probabilities to log odds."""
    return np.log(p / (1 - p))


def logit_se(p, n):
    """Approximate the standard error of log odds."""
    return np.sqrt(1 / (n * p) + 1 / (n * (1 - p)))


def make_stan_data(p1_mean, p2_mean, n_per_estimate):
    """Create Stan data on the log-odds scale."""
    return {
        "N1": len(p1_mean),
        "p1_logodds": logit(p1_mean).tolist(),
        "p1_logodds_se": logit_se(p1_mean, n_per_estimate).tolist(),
        "N2": len(p2_mean),
        "p2_logodds": logit(p2_mean).tolist(),
        "p2_logodds_se": logit_se(p2_mean, n_per_estimate).tolist(),
    }


def summarize_draws(draws, parameters, interval=0.90):
    """Summarize posterior draws."""
    lower_q = (1 - interval) / 2
    upper_q = 1 - lower_q

    rows = []

    for parameter in parameters:
        values = draws[parameter].to_numpy()

        rows.append({
            "parameter": parameter,
            "mean": values.mean(),
            "sd": values.std(),
            "lower_90": np.quantile(values, lower_q),
            "upper_90": np.quantile(values, upper_q),
        })

    return pd.DataFrame(rows)


####################
# Stan Model
####################

stan_code = """
data {
  int<lower=1> N1;
  int<lower=1> N2;

  vector[N1] p1_logodds;
  vector<lower=0>[N1] p1_logodds_se;

  vector[N2] p2_logodds;
  vector<lower=0>[N2] p2_logodds_se;
}

parameters {
  real alpha1;
  real alpha2;
}

transformed parameters {
  real<lower=0, upper=1> theta1;
  real<lower=0, upper=1> theta2;

  theta1 = inv_logit(alpha1);
  theta2 = inv_logit(alpha2);
}

model {
  p1_logodds ~ normal(alpha1, p1_logodds_se);
  p2_logodds ~ normal(alpha2, p2_logodds_se);

  theta1 ~ beta(2, 2);
  theta2 ~ beta(2, 2);
}

generated quantities {
  real<lower=0, upper=1> p_friend;

  p_friend = theta1 * theta2;
}
"""


####################
# Fit Model
####################

with open("friends_logodds.stan", "w") as file:
    file.write(stan_code)

stan_data = make_stan_data(
    p1_mean=p1_mean,
    p2_mean=p2_mean,
    n_per_estimate=N_PER_ESTIMATE,
)

model = csp.CmdStanModel(stan_file="friends_logodds.stan")

fit = model.sample(
    data=stan_data,
    chains=4,
    iter_warmup=1000,
    iter_sampling=1000,
    seed=123,
)


####################
# Posterior Summary
####################

draws = fit.draws_pd()

summary = summarize_draws(
    draws=draws,
    parameters=["theta1", "theta2", "p_friend"],
)

display(summary)


####################
# Manuscript Values
####################

for name in ["theta1", "theta2", "p_friend"]:
    values = draws[name].to_numpy()
    mean = values.mean()
    lower = np.quantile(values, 0.05)
    upper = np.quantile(values, 0.95)

    print(
        f"{name}: mean = {mean:.3f}, "
        f"90% posterior interval = ({lower:.3f}, {upper:.3f})"
    )

In [ ]:
import seaborn as sns
# plot_prevalence
make_aesthetic()

###############
# Data
###############

_ORDER  = ["any_ssl", "personal_psychological", "societal_conventional", "moral"]
_LABELS = ["Any SSL", "Personal", "Conventional", "Moral"]
_COLORS = ["#666666",
           DOMAIN_COLORS["personal_psychological"],
           DOMAIN_COLORS["societal_conventional"],
           DOMAIN_COLORS["moral"]]
_pv = prevalence_table.set_index("outcome")

###############
# Plot
###############
def plot_prevalence(pv, order, labels, colors):
    fig, ax = plt.subplots(figsize=(9, 6))
    for yi, (dom, lbl, col) in enumerate(zip(order, labels, colors)):
        r   = pv.loc[dom]
        pct = r["weighted_percent"]
        ax.barh(yi, pct, height=0.55, color=col, alpha=0.88, zorder=2)
        ax.errorbar(pct, yi,
                    xerr=[[pct - r["weighted_ci_low_percent"]],
                          [r["weighted_ci_high_percent"] - pct]],
                    fmt="none", color="#333333", capsize=4, linewidth=1.1, zorder=3)
        ax.text(pct + 2, yi+0.1, f"{pct:.1f}%", va="center")
    sns.despine(ax=ax, left=True)
    ax.set_yticks(range(len(order)))
    ax.set_yticklabels(labels)
    ax.set_xlabel("Weighted % of respondents  (95% CI)")
    ax.set_xlim(0, 105)
    ax.tick_params(axis="y", length=0)
    plt.title("Synthetic Social Learning (SSL) Prevalence\nAmong Chatbot Users")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "prevalence.pdf", bbox_inches="tight")
    plt.show()

plot_prevalence(_pv, _ORDER, _LABELS, _COLORS)


## 4  Motivations

In [ ]:
# analysis_motivation_crosstabs
motivation_options = {
    "practical": [
        "Nobody was available at the time",
        "I needed a response right away",
        "I can think out loud without needing to pre-organize my thoughts",
        "I wanted an unbiased opinion",
    ],
    "productive": [
        "I thought using a chatbot would save me time",
        "I thought using a chatbot would save me effort",
        "I thought I would learn something",
        "I thought using a chatbot would produce something valuable",
    ],
    "social": [
        "I wanted someone to talk to",
        "I didn't have a person I could talk to about this in my life",
        "I wanted AI to provide a particular perspective I didn't have access to",
        "I didn’t want to burden my friends or family",
    ],
    "entertain": [
        "I was bored",
        "I was curious what the chatbot would say",
        "I was experimenting with different chatbots",
        "I wanted to be distracted",
    ],
}

domain_specs = {
    "personal_psychological": {"prefix": "personal", "used_col": "used_personal_ssl"},
    "societal_conventional": {"prefix": "con", "used_col": "used_conventional_ssl"},
    "moral": {"prefix": "moral", "used_col": "used_moral_ssl"},
}

motivation_rows = []
respondent_factor_rows = []
for domain, spec in domain_specs.items():
    domain_df = df_analysis[df_analysis[spec["used_col"]].eq(1)].copy()
    denom = len(domain_df)
    weighted_denom = domain_df[REPORT_WEIGHT_COL].sum()
    for bucket, options in motivation_options.items():
        col = f"{spec['prefix']}_{bucket}"
        for option in options:
            selected = domain_df[col].fillna("").astype(str).str.contains(re.escape(option), regex=True)
            svy_est = svy_prop_level_row(domain_df.assign(_selected=selected.astype(int)), "_selected", level=1, weight_col=REPORT_WEIGHT_COL)
            motivation_rows.append({
                "domain": domain,
                "general_bucket": bucket,
                "specific_factor": option,
                "n_domain_users": denom,
                "raw_count": int(selected.sum()),
                "percent_of_domain_users": selected.mean() * 100 if denom else np.nan,
                "weighted_count": domain_df.loc[selected, REPORT_WEIGHT_COL].sum(),
                "weighted_percent_of_domain_users": svy_est["estimate"] * 100,
                "weighted_se": svy_est["se"],
                "weighted_ci_low": svy_est["ci_low"],
                "weighted_ci_high": svy_est["ci_high"],
                "weighted_ci_low_percent": svy_est["ci_low"] * 100,
                "weighted_ci_high_percent": svy_est["ci_high"] * 100,
            })

            selected_ids = domain_df.loc[selected, ["ResponseId", REPORT_WEIGHT_COL]].copy()
            selected_ids["domain"] = domain
            selected_ids["general_bucket"] = bucket
            selected_ids["specific_factor"] = option
            respondent_factor_rows.append(selected_ids)

motivation_crosstab_long = pd.DataFrame(motivation_rows)
motivation_crosstab_long["percent_of_domain_users"] = motivation_crosstab_long["percent_of_domain_users"].round(1)
motivation_crosstab_long["weighted_percent_of_domain_users"] = motivation_crosstab_long["weighted_percent_of_domain_users"].round(1)

respondent_factor_long = pd.concat(respondent_factor_rows, ignore_index=True) if respondent_factor_rows else pd.DataFrame()

# RQ tables exclude the explicit non-motivation option so bucket rankings reflect substantive factors.
substantive_factor_long = respondent_factor_long[
    respondent_factor_long["specific_factor"].ne("None of these reasons applies")
].copy()
substantive_crosstab = motivation_crosstab_long[
    motivation_crosstab_long["specific_factor"].ne("None of these reasons applies")
].copy()

# RQ1: What general bucket is most common overall? Each selected specific factor counts.
rq1_bucket_overall = (
    substantive_factor_long
    .groupby("general_bucket")
    .agg(raw_count=("ResponseId", "size"), weighted_count=(REPORT_WEIGHT_COL, "sum"))
    .reset_index()
    .sort_values("weighted_count", ascending=False)
)
rq1_bucket_overall["raw_percent"] = rq1_bucket_overall["raw_count"] / rq1_bucket_overall["raw_count"].sum() * 100
rq1_svy = svy_tabulate_df(substantive_factor_long, rowvar="general_bucket", weight_col=REPORT_WEIGHT_COL, units="percent")
rq1_svy = rq1_svy.rename(columns={
    "row": "general_bucket",
    "svy_percent": "weighted_percent",
    "svy_percent_se": "weighted_percent_se",
    "svy_percent_ci_low": "weighted_ci_low_percent",
    "svy_percent_ci_high": "weighted_ci_high_percent",
})
rq1_bucket_overall = rq1_bucket_overall.drop(columns=["weighted_percent"], errors="ignore").merge(
    rq1_svy[["general_bucket", "weighted_percent", "weighted_percent_se", "weighted_ci_low_percent", "weighted_ci_high_percent"]],
    on="general_bucket",
    how="left",
).sort_values("weighted_percent", ascending=False)

# RQ2: What general bucket is most common for each domain? Each selected specific factor counts.
rq2_grouped = (
    substantive_factor_long
    .groupby(["domain", "general_bucket"])
    .agg(raw_count=("ResponseId", "size"), weighted_count=(REPORT_WEIGHT_COL, "sum"))
    .reset_index()
)
rq2_grouped["raw_percent_within_domain"] = rq2_grouped["raw_count"] / rq2_grouped.groupby("domain")["raw_count"].transform("sum") * 100
rq2_svy_rows = []
for domain, domain_rows in substantive_factor_long.groupby("domain"):
    tab = svy_tabulate_df(domain_rows, rowvar="general_bucket", weight_col=REPORT_WEIGHT_COL, units="percent")
    tab = tab.rename(columns={
        "row": "general_bucket",
        "svy_percent": "weighted_percent_within_domain",
        "svy_percent_se": "weighted_percent_se",
        "svy_percent_ci_low": "weighted_ci_low_percent",
        "svy_percent_ci_high": "weighted_ci_high_percent",
    })
    tab["domain"] = domain
    rq2_svy_rows.append(tab[["domain", "general_bucket", "weighted_percent_within_domain", "weighted_percent_se", "weighted_ci_low_percent", "weighted_ci_high_percent"]])
rq2_svy = pd.concat(rq2_svy_rows, ignore_index=True) if rq2_svy_rows else pd.DataFrame()
rq2_bucket_by_domain = rq2_grouped.merge(rq2_svy, on=["domain", "general_bucket"], how="left")
rq2_bucket_by_domain = rq2_bucket_by_domain.sort_values(["domain", "weighted_percent_within_domain"], ascending=[True, False])

# RQ3: What specific factors are most common?
rq3_specific_factors = substantive_crosstab.sort_values(["domain", "weighted_count"], ascending=[True, False])

for table in [rq1_bucket_overall, rq2_bucket_by_domain]:
    for col in table.columns:
        if "percent" in col:
            table[col] = table[col].round(1)

print("RQ1: Most common general motivation buckets overall")
display(rq1_bucket_overall)
print("RQ2: Motivation buckets by SSL domain")
display(rq2_bucket_by_domain)
print("RQ3: Specific motivation factors by SSL domain")
display(rq3_specific_factors)

motivation_crosstab_long.to_csv(OUTPUT_DIR / "ssl_motivation_crosstab_long.csv", index=False)
respondent_factor_long.to_csv(OUTPUT_DIR / "ssl_motivation_respondent_factor_long.csv", index=False)
substantive_factor_long.to_csv(OUTPUT_DIR / "ssl_motivation_substantive_respondent_factor_long.csv", index=False)
rq1_bucket_overall.to_csv(OUTPUT_DIR / "rq1_motivation_bucket_overall.csv", index=False)
rq2_bucket_by_domain.to_csv(OUTPUT_DIR / "rq2_motivation_bucket_by_domain.csv", index=False)
rq3_specific_factors.to_csv(OUTPUT_DIR / "rq3_specific_motivation_factors.csv", index=False)

# Backward-compatible names from the earlier section.
motivation_table = motivation_crosstab_long.copy()
motivation_top = motivation_crosstab_long.sort_values(["domain", "weighted_count"], ascending=[True, False]).groupby("domain").head(10)


In [ ]:

import textwrap
from helpers import make_aesthetic
import matplotlib.lines as mlines
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams.update({
    "font.family":       "sans-serif",
    "font.sans-serif":   ["Helvetica", "Arial", "DejaVu Sans"],
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "pdf.fonttype":      42,
    "ps.fonttype":       42,
})

####################
# Font hierarchy
####################

BASE = 11
FONT = {
    "small":  BASE,          # 11 — y-axis labels, tick labels, legend entries
    "medium": BASE * 1.25,   # ~14 — axis labels, bucket headers, legend title
    "large":  BASE * 1.75,   # ~19 — plot title
}

COLORS = make_aesthetic(font_scale=1.2, bold_title=True)

DOMAIN_ORDER_RAW = ["personal_psychological", "societal_conventional", "moral"]
DOMAIN_DISPLAY   = {"personal_psychological": "Personal",
                    "societal_conventional":  "Conventional",
                    "moral":                  "Moral"}
DOMAIN_MARKERS   = {"personal_psychological": "o",
                    "societal_conventional":  "D",
                    "moral":                  "^"}
# Domain colors from make_aesthetic default palette (indices 0, 4, 7)
# Bucket colors occupy indices 5, 6, 8, 12 — no conflict
DOMAIN_COLORS    = {"personal_psychological": COLORS[0],   # #00A896 teal-green
                    "societal_conventional":  COLORS[7],   # #020887 dark-blue
                    "moral":                  COLORS[4]}   # #F45B69 coral-red
DOMAIN_SIZES     = {"personal_psychological": 7,
                    "societal_conventional":  6,
                    "moral":                  7}

BUCKET_ORDER   = ["practical", "productive", "social", "entertain"]
BUCKET_DISPLAY = {"practical":  "Practical",
                  "productive": "Productivity",
                  "social":     "Social",
                  "entertain":  "Entertainment"}
BUCKET_COLORS  = dict(zip(BUCKET_ORDER, [COLORS[5], COLORS[6], COLORS[8], COLORS[12]]))

_factor_to_bucket = (
    substantive_crosstab[["specific_factor", "general_bucket"]]
    .drop_duplicates()
    .set_index("specific_factor")["general_bucket"]
    .to_dict()
)

_Y_OFF   = {d: (i - 1) * 0.26 for i, d in enumerate(DOMAIN_ORDER_RAW)}
GAP_SIZE = 1.2


####################
# Helper functions
####################

def _format_factor_label(factor, display_override=None):
    if display_override and factor in display_override:
        return display_override[factor]
    raw = factor.replace("_", " ")
    return raw[0].upper() + raw[1:]


def _build_grouped_order(data, y_col, est_col, bucket_lookup):
    """Sort factors within each bucket by mean estimate (ascending), insert gaps between buckets."""
    mean_by_factor = data.groupby(y_col)[est_col].mean()
    positions, bucket_spans = {}, {}
    cursor = 0.0
    buckets_seen = [b for b in BUCKET_ORDER
                    if any(bucket_lookup.get(f) == b for f in mean_by_factor.index)]
    for bi, bucket in enumerate(buckets_seen):
        factors = [f for f in mean_by_factor.index if bucket_lookup.get(f) == bucket]
        sorted_factors = sorted(factors, key=lambda f: mean_by_factor[f])
        y_first = cursor
        for factor in sorted_factors:
            positions[factor] = cursor
            cursor += 1.0
        bucket_spans[bucket] = (y_first, cursor - 1.0)
        if bi < len(buckets_seen) - 1:
            cursor += GAP_SIZE
    return positions, bucket_spans, cursor - 1.0


####################
# Plot function
####################

def draw_dot_plot(data, y_col, est_col, ci_lo_col, ci_hi_col,
                  title, filename_stem, bucket_lookup,
                  factor_display=None, wrap=42):

    positions, bucket_spans, total_height = _build_grouped_order(
        data, y_col, est_col, bucket_lookup
    )
    factor_order    = sorted(positions, key=lambda f: positions[f])
    buckets_present = [b for b in BUCKET_ORDER if b in bucket_spans]
    n_factors       = len(factor_order)

    fig, ax = plt.subplots(
        figsize=(11, max(6, n_factors * 0.75 + len(buckets_present) * GAP_SIZE * 0.9))
    )
    fig.subplots_adjust(top=0.88)

    # Dots and CIs
    for dom in DOMAIN_ORDER_RAW:
        sub = data[data["domain"].eq(dom)]
        for factor, yi in positions.items():
            row = sub[sub[y_col].eq(factor)]
            if row.empty:
                continue
            est = float(row[est_col].values[0])
            lo  = float(row[ci_lo_col].values[0])
            hi  = float(row[ci_hi_col].values[0])
            ax.errorbar(
                est, yi + _Y_OFF[dom],
                xerr=[[est - lo], [hi - est]],
                fmt=DOMAIN_MARKERS[dom],
                color=DOMAIN_COLORS[dom],
                markersize=DOMAIN_SIZES[dom],
                capsize=2.5, linewidth=1.1, elinewidth=0.9, zorder=3,
            )

    # Per-row gridlines
    for factor, yi in positions.items():
        ax.axhline(yi, color="#f2f2f2", linewidth=0.5, zorder=1)

    # Y-axis labels
    ytick_positions = [positions[f] for f in factor_order]
    display_labels  = [
        textwrap.fill(_format_factor_label(f, factor_display), wrap)
        for f in factor_order
    ]
    ax.set_yticks(ytick_positions)
    tick_texts = ax.set_yticklabels(display_labels, fontsize=FONT["small"], ha="right")
    for tick_lbl in tick_texts:
        tick_lbl.set_color("#1a1a1a")

    ax.autoscale(axis="x")
    x_lo, x_hi = ax.get_xlim()

    # Dashed rounded rectangles
    rect_right = x_hi * 0.98
    for bucket in buckets_present:
        y_first, y_last = bucket_spans[bucket]
        rect = mpatches.FancyBboxPatch(
            (x_lo * 0.99, y_first - 0.48),
            width=rect_right - x_lo * 0.99,
            height=(y_last - y_first) + 0.96,
            boxstyle="round,pad=0.0,rounding_size=0.15",
            linewidth=0.8, edgecolor="#aaaaaa",
            facecolor="none", linestyle="--",
            zorder=0, clip_on=False,
        )
        ax.add_patch(rect)

    # Bucket labels above each rectangle
    for bucket in buckets_present:
        y_first, _ = bucket_spans[bucket]
        ax.text(
            x_lo * 0.99, y_first - 0.62,
            f"\nMotivation Type: {BUCKET_DISPLAY[bucket]}",
            va="bottom", ha="left",
            fontsize=FONT["medium"], fontweight="normal", fontstyle="italic",
            color="#1a1a1a", clip_on=False,
        )

    # Legend — top right, aligned with title
    shape_handles = [
        mlines.Line2D([], [], marker=DOMAIN_MARKERS[d],
                      color=DOMAIN_COLORS[d],
                      linestyle="none",
                      markersize=DOMAIN_SIZES[d] + 1,
                      label=DOMAIN_DISPLAY[d])
        for d in DOMAIN_ORDER_RAW
    ]
    ax.legend( #here
    handles=shape_handles,
    loc="upper left",
    bbox_to_anchor=(0.5, 0.93),
    frameon=True,
    ncol=3,
    fontsize=FONT["small"],
    framealpha=1.0,
    facecolor="#f2f2f2",
    edgecolor="#bdbdbd",
    title="Synthetic Social Learning domain",
    title_fontsize=FONT["medium"],
)
    # Title — flush left, top-aligned at same y as legend
    ax.set_title("", pad=42)
    ax.text(
        0, 1.08, title,
        transform=ax.transAxes,
        fontsize=FONT["large"], fontweight="bold",
        va="top", ha="left",
    )

    # Axis formatting
    ax.xaxis.tick_top()
    ax.xaxis.set_label_position("top")
    ax.set_xlabel("Weighted % of domain users", labelpad=10, fontsize=FONT["medium"])
    ax.tick_params(axis="x", labelsize=FONT["small"], top=True, length=3, width=0.6)
    ax.tick_params(axis="y", left=False, pad=6)
    ax.spines[["left", "bottom", "right"]].set_visible(False)
    ax.spines["top"].set_color("#cccccc")
    ax.spines["top"].set_linewidth(0.6)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f}%"))
    ax.set_ylim(-1.4, total_height + 1.4)
    ax.invert_yaxis()

    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.savefig(OUTPUT_DIR / f"{filename_stem}.pdf", bbox_inches="tight")
    plt.show()


####################
# Plot
####################

draw_dot_plot(
    data=substantive_crosstab,
    y_col="specific_factor",
    est_col="weighted_percent_of_domain_users",
    ci_lo_col="weighted_ci_low_percent",
    ci_hi_col="weighted_ci_high_percent",
    title="Motivations for Synthetic Social Learning By Domain",
    filename_stem="motivation_factor_dotplot",
    bucket_lookup=_factor_to_bucket,
    wrap=42,
)


# ── Alluvial (filled Sankey) ──────────────────────────────────────────────
def draw_alluvial(title, filename_stem):
    """Filled Sankey: motivation buckets (left) → SSL domains (right)."""
    from matplotlib.patches import Rectangle

    flow_df = (
        substantive_factor_long
        .drop_duplicates(subset=["ResponseId", "domain", "general_bucket"])
        .groupby(["domain", "general_bucket"])[REPORT_WEIGHT_COL]
        .sum().reset_index().rename(columns={REPORT_WEIGHT_COL: "flow"})
    )
    bt = {b: float(flow_df[flow_df["general_bucket"].eq(b)]["flow"].sum()) for b in BUCKET_ORDER}
    dt = {d: float(flow_df[flow_df["domain"].eq(d)]["flow"].sum())          for d in DOMAIN_ORDER_RAW}

    def _positions(order, totals, gap_frac=0.10):
        total = sum(totals[k] for k in order); n = len(order)
        gap = gap_frac * total / n; denom = total + (n-1)*gap
        nodes, cursor = {}, 0.0
        for k in order:
            h = totals[k]/denom
            nodes[k] = {"y_bot": cursor, "y_top": cursor+h, "h": h, "fill": cursor}
            cursor += h + gap/denom
        return nodes

    ln = _positions(BUCKET_ORDER, bt)
    rn = _positions(DOMAIN_ORDER_RAW, dt)

    X_L, X_R, BAR_W = 0.12, 0.87, 0.022
    T = np.linspace(0,1,300)
    XA=X_L+BAR_W; XC1=XA+(X_R-XA)*0.40; XC2=XA+(X_R-XA)*0.60
    XC=(1-T)**3*XA+3*(1-T)**2*T*XC1+3*(1-T)*T**2*XC2+T**3*X_R

    make_aesthetic()
    fig, ax = plt.subplots(figsize=(11, 7.5))
    fig.subplots_adjust(top=0.88, left=0.02, right=0.98)
    ax.set_xlim(0,1); ax.set_ylim(-0.02,1.05); ax.axis("off")

    for bucket in BUCKET_ORDER:
        for domain_raw in DOMAIN_ORDER_RAW:
            hit = flow_df[flow_df["general_bucket"].eq(bucket)&flow_df["domain"].eq(domain_raw)]
            if hit.empty: continue
            flow=float(hit["flow"].values[0])
            h_l=(flow/bt[bucket])*ln[bucket]["h"]; h_r=(flow/dt[domain_raw])*rn[domain_raw]["h"]
            y0b,y0t=ln[bucket]["fill"],ln[bucket]["fill"]+h_l
            y1b,y1t=rn[domain_raw]["fill"],rn[domain_raw]["fill"]+h_r
            ln[bucket]["fill"]+=h_l; rn[domain_raw]["fill"]+=h_r
            ax.fill_between(XC, y0b+(y1b-y0b)*T, y0t+(y1t-y0t)*T,
                            color=BUCKET_COLORS[bucket], alpha=0.55, lw=0, zorder=2)

    for bucket in BUCKET_ORDER:
        n=ln[bucket]
        ax.add_patch(Rectangle((X_L,n["y_bot"]),BAR_W,n["h"],color=BUCKET_COLORS[bucket],zorder=4,lw=0))
        ax.text(X_L-0.012,(n["y_bot"]+n["y_top"])/2,BUCKET_DISPLAY[bucket],
                ha="right",va="center",fontweight="bold",color=BUCKET_COLORS[bucket])
    for domain_raw in DOMAIN_ORDER_RAW:
        n=rn[domain_raw]
        ax.add_patch(Rectangle((X_R,n["y_bot"]),BAR_W,n["h"],color="#1a1a1a",zorder=4,lw=0,alpha=0.85))
        ax.text(X_R+BAR_W+0.012,(n["y_bot"]+n["y_top"])/2,DOMAIN_DISPLAY[domain_raw],
                ha="left",va="center",fontweight="bold",color="#1a1a1a")

    if title:
        ax.text(0,1.06,title,transform=ax.transAxes,fontweight="bold",va="top",ha="left")
    plt.savefig(OUTPUT_DIR/f"{filename_stem}.pdf",bbox_inches="tight")
    plt.show()


In [ ]:
# analysis_motivation_bars_quadrants
make_aesthetic(font_scale=1.0)
plt.rcParams["figure.constrained_layout.use"] = False

# Font hierarchy — change BASE to scale everything proportionally
BASE  = 13
FS    = {"xs": BASE * 0.75, "sm": BASE, "md": BASE * 1.15, "lg": BASE * 1.4}

_PAL = make_aesthetic(font_scale=1.0)
_DOM_COLORS = {
    "personal_psychological": _PAL[0],
    "societal_conventional":  _PAL[7],
    "moral":                  _PAL[4],
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.subplots_adjust(hspace=0.28, wspace=0.58,
                    left=0.30, right=0.98,
                    top=0.90, bottom=0.04)

for ax, bucket in zip(axes.flat, BUCKET_ORDER):
    sub = substantive_crosstab[substantive_crosstab["general_bucket"].eq(bucket)]
    factor_order = (
        sub.groupby("specific_factor")["weighted_percent_of_domain_users"]
        .mean().sort_values(ascending=False).index.tolist()
    )

    for dom in DOMAIN_ORDER_RAW:
        dom_data = sub[sub["domain"].eq(dom)]
        for yi, f in enumerate(factor_order):
            row = dom_data[dom_data["specific_factor"].eq(f)]
            if row.empty:
                continue
            est = float(row["weighted_percent_of_domain_users"].values[0])
            lo  = float(row["weighted_ci_low_percent"].values[0])
            hi  = float(row["weighted_ci_high_percent"].values[0])
            ax.errorbar(
                est, yi + _Y_OFF[dom],
                xerr=[[est - lo], [hi - est]],
                fmt=DOMAIN_MARKERS[dom], color=_DOM_COLORS[dom],
                markersize=DOMAIN_SIZES[dom],
                capsize=2.5, linewidth=1.0, elinewidth=0.85, zorder=3)

    for yi in range(len(factor_order)):
        ax.axhline(yi, color="#f2f2f2", linewidth=0.4, zorder=1)

    ax.set_yticks(range(len(factor_order)))
    ax.set_yticklabels([textwrap.fill(f, 32) for f in factor_order],
                       fontsize=FS["sm"])
    ax.set_xlim(0, 80)
    ax.invert_yaxis()
    ax.xaxis.tick_top()
    ax.xaxis.set_label_position("top")
    ax.set_xlabel("Weighted % of Domain Respondents (95% CI)", fontsize=FS["sm"], labelpad=4)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f}%"))
    ax.tick_params(axis="x", labelsize=FS["xs"], labelrotation=0, length=2, width=0.5)
    ax.tick_params(axis="y", length=0, pad=4)
    ax.spines[["left", "bottom", "right"]].set_visible(False)
    ax.spines["top"].set_color("#d0d0d0")
    ax.spines["top"].set_linewidth(0.5)
    ax.set_ylim(len(factor_order) - 0.6, -0.55)

    ax.text(0.03, 0.96, f"Motivation: {BUCKET_DISPLAY[bucket]}",
            transform=ax.transAxes,
            fontsize=FS["md"], fontstyle="italic", va="top", ha="left")

axes[0, 0].legend(
    handles=[mlines.Line2D([], [], marker=DOMAIN_MARKERS[d], color=_DOM_COLORS[d],
                           linestyle="none", markersize=7, label=DOMAIN_DISPLAY[d])
             for d in DOMAIN_ORDER_RAW],
    title="Domain",
    loc="lower right", frameon=True,
    fontsize=FS["sm"], title_fontsize=FS["sm"],
    framealpha=1.0, edgecolor="#bbbbbb", facecolor="#efefef")

plt.suptitle("Motivations by Synthetic Social Learning Domain\n",
             fontsize=FS["lg"], fontweight="bold", y=1.05) #here
plt.savefig(OUTPUT_DIR / "ssl_motivation_quadrants.pdf", bbox_inches="tight")
plt.savefig(OUTPUT_DIR / "ssl_motivation_quadrants.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# alluvial
draw_alluvial(title="", filename_stem="motivation_alluvial")


## 5  Outcomes

In [ ]:
# outcomes_satisfaction_rec

###############
# Recoding / helpers
###############
SAT_REC_MAP = {
    "personal_psychological": {"satisfied": "personal_satisfied", "recfriend": "personal_recfriend"},
    "societal_conventional":  {"satisfied": "con_satisfied",      "recfriend": "con_recfriend"},
    "moral":                  {"satisfied": "moral_satisfied",     "recfriend": "moral_recfriend"},
}
USED_COL = {
    "personal_psychological": "used_personal_ssl",
    "societal_conventional":  "used_conventional_ssl",
    "moral":                  "used_moral_ssl",
}

def compute_sat_rec(df_a):
    for domain, items in SAT_REC_MAP.items():
        for metric, col in items.items():
            num_col = col + "_num"
            if num_col not in df_a.columns:
                df_a[num_col] = df_a[col].apply(extract_leading_number)
    rows = []
    for domain_raw, items in SAT_REC_MAP.items():
        domain_df = df_a[df_a[USED_COL[domain_raw]].eq(1)].copy()
        for metric, col in items.items():
            num_col = col + "_num"
            est = svy_mean_row(domain_df, num_col)
            rows.append({"domain": domain_raw, "metric": metric,
                         "n": int(domain_df[num_col].notna().sum()), **est})
    return pd.DataFrame(rows)

###############
# Compute
###############
sat_rec_df = compute_sat_rec(df_analysis)
display(sat_rec_df.round(2))
sat_rec_df.to_csv(OUTPUT_DIR / "sat_rec.csv", index=False)


In [ ]:
import seaborn as sns

make_aesthetic()

###############
# Config
###############

METRIC_CONFIG = {
    "satisfied": {
        "label": "Satisfied with experience",
        "offset": -0.18,
        "color": COLORS[2],
    },
    "recfriend": {
        "label": "Would recommend a friend use LLM this way",
        "offset": 0.18,
        "color": COLORS[4],
    },
}

METRIC_ORDER_RAW = ["satisfied", "recfriend"]


###############
# Plot
###############

def plot_outcomes(df):
    _rd = df.copy()
    _rd["Domain"] = _rd["domain"].map(DOMAIN_DISPLAY)

    domain_order = [DOMAIN_DISPLAY[d] for d in DOMAIN_ORDER_RAW]

    fig, ax = plt.subplots(figsize=(9, 6))

    for yi, domain in enumerate(domain_order):
        for metric_raw in METRIC_ORDER_RAW:
            metric_label = METRIC_CONFIG[metric_raw]["label"]
            metric_offset = METRIC_CONFIG[metric_raw]["offset"]
            metric_color = METRIC_CONFIG[metric_raw]["color"]

            row = _rd[_rd["Domain"].eq(domain) & _rd["metric"].eq(metric_raw)]

            if row.empty:
                continue

            row = row.iloc[0]
            y = yi + metric_offset

            est = float(row["estimate"])
            lo = float(row["ci_low"])
            hi = float(row["ci_high"])

            ax.barh(
                y,
                est - 1,
                left=1,
                height=0.30,
                color=metric_color,
                alpha=0.88,
                label=metric_label if yi == 0 else "_nolegend_",
            )

            ax.errorbar(
                est,
                y,
                xerr=[[est - lo], [hi - est]],
                fmt="none",
                color="#333333",
                capsize=3,
                linewidth=1.0,
                zorder=3,
            )

            ax.text(
                min(hi + 0.12, 8.85),
                y,
                f"M = {est:.1f}",
                va="center",
                ha="left",
                fontsize=10,
            )

    ax.set_yticks(range(len(domain_order)))
    ax.set_yticklabels(domain_order)

    ax.set_xlabel("Score (1–9 scale)")
    ax.set_ylabel("")
    ax.set_xlim(1, 9)
    ax.set_xticks(range(1, 10))

    ax.set_title(
        "SSL Experience: User Satisfaction and\nWillingness to Recommend",
        fontweight="bold",
    )

    ax.legend(
        title="",
        frameon=False,
        loc="upper right",
        bbox_to_anchor=(1.1, -0.1), #here,
        ncol=2,
    )

    sns.despine(ax=ax, left=True)

    fig.subplots_adjust(
        left=0.18,
        right=0.98,
        bottom=0.12,
        top=0.90,
    )

    plt.savefig(OUTPUT_DIR / "outcomes_sat_rec.pdf", bbox_inches="tight")
    plt.show()


plot_outcomes(sat_rec_df)

In [ ]:
# analysis_ssl_intensity
intensity_cols = {
    "personal_ssl_intensity": "domain_p_freq",
    "societal_conventional_ssl_intensity": "domain_c_freq",
    "moral_ssl_intensity": "domain_e_freq",
}

for new_col, source_col in intensity_cols.items():
    df[new_col] = df[source_col].apply(frequency_code)
    df_analysis[new_col] = df_analysis[source_col].apply(frequency_code)

domain_intensity_cols = list(intensity_cols.keys())
df["ssl_intensity_total"] = df[domain_intensity_cols].sum(axis=1, min_count=1)
df_analysis["ssl_intensity_total"] = df_analysis[domain_intensity_cols].sum(axis=1, min_count=1)

intensity_summary = []
for col in domain_intensity_cols + ["ssl_intensity_total"]:
    valid = df_analysis[col].dropna()
    svy_est = svy_mean_row(df_analysis, col, REPORT_WEIGHT_COL)
    intensity_summary.append({
        "outcome": col,
        "n_valid": len(valid),
        "mean": valid.mean(),
        "sd": valid.std(),
        "weighted_mean": svy_est["estimate"],
        "weighted_se": svy_est["se"],
        "weighted_ci_low": svy_est["ci_low"],
        "weighted_ci_high": svy_est["ci_high"],
        "min": valid.min(),
        "max": valid.max(),
    })

intensity_summary = pd.DataFrame(intensity_summary)
display(intensity_summary)
intensity_summary.to_csv(OUTPUT_DIR / "ssl_intensity_summary.csv", index=False)

intensity_distribution_rows = []
for col in domain_intensity_cols + ["ssl_intensity_total"]:
    dist = df_analysis.groupby(col, dropna=False).agg(
        raw_count=("ResponseId", "size"),
        weighted_count=(REPORT_WEIGHT_COL, "sum"),
    ).reset_index().rename(columns={col: "intensity"})
    dist["outcome"] = col
    dist["raw_percent"] = dist["raw_count"] / dist["raw_count"].sum() * 100
    dist["weighted_percent"] = dist["weighted_count"] / dist["weighted_count"].sum() * 100
    intensity_distribution_rows.append(dist)

intensity_distribution = pd.concat(intensity_distribution_rows, ignore_index=True)
intensity_distribution["raw_percent"] = intensity_distribution["raw_percent"].round(1)
intensity_distribution["weighted_percent"] = intensity_distribution["weighted_percent"].round(1)
display(intensity_distribution)
intensity_distribution.to_csv(OUTPUT_DIR / "ssl_intensity_distribution.csv", index=False)


## 6  Regressions

Four models: **binary** (`used_any_ssl`) and **intensity** (`ssl_intensity_total`) × **unweighted** and **survey-design weighted**.

In [ ]:
# regression_functions

###############
# Predictor labels (with explicit reference categories)
###############
PREDICTOR_LABELS = {
    "age_num_z":               "Age (z)",
    "gender_male":             "Male  [ref: Female]",
    "gender_other":            "Gender: other  [ref: Female]",
    "race_white":              "White  [ref: other race]",
    "race_black":              "Black  [ref: other race]",
    "pid_republican":          "Republican  [ref: Independent]",
    "pid_democrat":            "Democrat  [ref: Independent]",
    "income_ord_z":            "Income (z)",
    "college_degree":          "College degree",
    "employed":                "Employed  [ref: not]",
    "retired":                 "Retired  [ref: not]",
    "ideo_num_z":              "Ideology: conservative (z)",
    "use_freq_code_z":         "LLM use frequency (z)",
    "aias4_score_z":           "AI attitudes (z)",
    "anthrotech_score_z":      "AI anthropomorphism (z)",
    "sias4_score_z":           "Social anxiety (z)",
    "lsns6_score_z":           "Social network size (z)",
    "tipi_extraversion_z":     "Extraversion (z)",
    "tipi_agreeableness_z":    "Agreeableness (z)",
    "tipi_conscientiousness_z":"Conscientiousness (z)",
    "tipi_emotional_stability_z":"Emotional stability (z)",
    "tipi_openness_z":         "Openness (z)",
    "ai_social_use_index_z":   "AI social use index (z)",
}

def label_pred(term):
    return PREDICTOR_LABELS.get(
        term, term.replace("_z","").replace("_"," ").title())


###############
# MultiModelHandler
###############
from statsmodels_handler import StatsmodelsHandler

class MultiModelHandler:
    """Overlay weighted + unweighted on one forest plot.
    Color = significance direction; shape = model.
    """
    _INTERCEPTS = {"const", "Intercept", "_intercept_"}
    _SHAPES     = ["o", "D", "s", "^"]

    def __init__(self, models: dict):
        """models: {label: statsmodels_result | DataFrame{term,coef,ci_low,ci_high,p}}"""
        self.frames = {}
        for label, m in models.items():
            if isinstance(m, pd.DataFrame):
                self.frames[label] = m[["term","coef","ci_low","ci_high","p"]].copy()
            else:
                conf = m.conf_int()
                self.frames[label] = pd.DataFrame({
                    "term":    m.params.index,
                    "coef":    m.params.values,
                    "ci_low":  conf[0].values,
                    "ci_high": conf[1].values,
                    "p":       m.pvalues.values,
                })

    def plot(self, exp=False, drop_intercept=True,
             clean_var_name=label_pred, figsize=(9, 8)):
        pal = make_aesthetic()
        C_POS, C_NEG, C_NS = pal[0], pal[6], "#aaaaaa"

        # Term order: sort by mean |coef| descending (largest effect at top after invert)
        ref_df  = next(iter(self.frames.values()))
        terms   = ref_df["term"].tolist()
        if drop_intercept:
            terms = [t for t in terms if t not in self._INTERCEPTS]

        mean_abs = {}
        for t in terms:
            vals = []
            for df in self.frames.values():
                row = df[df["term"].eq(t)]
                if not row.empty:
                    c = float(row["coef"].values[0])
                    vals.append(abs(np.exp(c) - 1 if exp else c))
            mean_abs[t] = np.mean(vals) if vals else 0
        terms = sorted(terms, key=lambda t: mean_abs[t])   # ascending so invert puts largest at top

        display_terms = [clean_var_name(t) for t in terms] if clean_var_name else terms
        threshold     = 1 if exp else 0
        n_models      = len(self.frames)
        y_offs        = [(i - (n_models - 1) / 2) * 0.20 for i in range(n_models)]

        fig, ax = plt.subplots(figsize=figsize)
        for (label, df), shape, yoff in zip(self.frames.items(), self._SHAPES, y_offs):
            for yi, term in enumerate(terms):
                row = df[df["term"].eq(term)]
                if row.empty:
                    continue
                c   = float(row["coef"].values[0])
                lo  = float(row["ci_low"].values[0])
                hi  = float(row["ci_high"].values[0])
                p   = float(row["p"].values[0])
                if exp:
                    c, lo, hi = np.exp(c), np.exp(lo), np.exp(hi)
                color = (C_POS if (p < 0.05 and c > threshold)
                         else C_NEG if (p < 0.05 and c < threshold)
                         else C_NS)
                ax.errorbar(c, yi + yoff,
                            xerr=[[max(0, c - lo)], [max(0, hi - c)]],
                            fmt=shape, color=color,
                            markersize=5, capsize=3, linewidth=0.9, elinewidth=0.75,
                            label=label if yi == 0 else "_nolegend_",
                            zorder=3)
        ax.axvline(threshold, color="#cccccc", linestyle="--", linewidth=0.7)
        ax.set_yticks(range(len(terms)))
        ax.set_yticklabels(display_terms)
        ax.invert_yaxis()
        ax.legend(frameon=False, loc="lower right")
        sns.despine(ax=ax)
        return fig, ax


###############
# Fitting functions
###############
def fit_logit(outcome_col, label):
    model_df = df_analysis[[outcome_col] + model_predictors].dropna().copy()
    y = model_df[outcome_col].astype(int)
    X = sm.add_constant(model_df[model_predictors], has_constant="add")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        result = sm.Logit(y, X).fit(disp=False, maxiter=200)
    conf = result.conf_int()
    tbl = pd.DataFrame({"outcome": label, "model": "unweighted",
                        "term": result.params.index,
                        "coef": result.params.values,
                        "or":   np.exp(result.params.values),
                        "ci_low": conf[0].values, "ci_high": conf[1].values,
                        "p": result.pvalues.values, "n": int(result.nobs)})
    return result, tbl


def fit_svy_logit(outcome_col, label):
    model_cols = [outcome_col, REPORT_WEIGHT_COL] + model_predictors
    model_df = df_analysis[model_cols].dropna().copy()
    model_df[outcome_col] = model_df[outcome_col].astype(int)
    sample = make_svy_sample(model_df, REPORT_WEIGHT_COL)
    result = sample.glm.fit(y=outcome_col, x=model_predictors,
                            family="binomial", link="logit", max_iter=200)
    tbl = result.to_polars().to_pandas().rename(
        columns={"estimate":"coef","conf_low":"ci_low","conf_high":"ci_high","p_value":"p"})
    tbl["outcome"] = label; tbl["model"] = "weighted"
    tbl["or"] = np.exp(tbl["coef"])
    tbl["term"] = tbl["term"].replace("_intercept_", "const")
    tbl["n"] = len(model_df)
    return tbl[["outcome","model","term","coef","or","ci_low","ci_high","p","n"]]


def fit_ols(outcome_col, label):
    model_df = df_analysis[[outcome_col] + model_predictors].dropna().copy()
    X = sm.add_constant(model_df[model_predictors], has_constant="add")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        result = sm.OLS(model_df[outcome_col], X).fit()
    conf = result.conf_int()
    tbl = pd.DataFrame({"outcome": label, "model": "unweighted",
                        "term": result.params.index,
                        "coef": result.params.values,
                        "ci_low": conf[0].values, "ci_high": conf[1].values,
                        "p": result.pvalues.values, "n": int(result.nobs)})
    return result, tbl


def fit_svy_ols(outcome_col, label):
    model_cols = [outcome_col, REPORT_WEIGHT_COL] + model_predictors
    model_df = df_analysis[model_cols].dropna().copy()
    sample = make_svy_sample(model_df, REPORT_WEIGHT_COL)
    result = sample.glm.fit(y=outcome_col, x=model_predictors,
                            family="gaussian", link="identity", max_iter=200)
    tbl = result.to_polars().to_pandas().rename(
        columns={"estimate":"coef","conf_low":"ci_low","conf_high":"ci_high","p_value":"p"})
    tbl["outcome"] = label; tbl["model"] = "weighted"
    tbl["term"] = tbl["term"].replace("_intercept_", "const")
    tbl["n"] = len(model_df)
    return tbl[["outcome","model","term","coef","ci_low","ci_high","p","n"]]


In [ ]:
# regression_df_vif_spearman
from scipy import stats
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.weightstats import DescrStatsW

###############
# Shared regression dataframe
###############
model_predictors = [
    "age_num_z", "gender_male", "gender_other", "race_white", "race_black",
    "pid_republican", "pid_democrat", "income_ord_z", "college_degree",
    "employed", "retired", "ideo_num_z",
    "use_freq_code_z", "aias4_score_z", "anthrotech_score_z",
    "sias4_score_z", "lsns6_score_z",
    "tipi_extraversion_z", "tipi_agreeableness_z", "tipi_conscientiousness_z",
    "tipi_emotional_stability_z", "tipi_openness_z", "ai_social_use_index_z",
]

LOGIT_OUTCOMES = {
    "any":          "used_any_ssl",
    "personal":     "used_personal_ssl",
    "conventional": "used_conventional_ssl",
    "moral":        "used_moral_ssl",
}

OLS_OUTCOMES = {
    "total_intensity":        "ssl_intensity_total",
    "personal_intensity":     "personal_ssl_intensity",
    "conventional_intensity": "societal_conventional_ssl_intensity",
    "moral_intensity":        "moral_ssl_intensity",
}

ALL_OUTCOMES = {**LOGIT_OUTCOMES, **OLS_OUTCOMES}
OUTCOME_LABELS = {
    "any": "Any SSL",
    "personal": "Any Personal SSL",
    "conventional": "Any Conventional SSL",
    "moral": "Any Moral SSL",
    "total_intensity": "SSL Intensity",
    "personal_intensity": "Personal SSL Intensity",
    "conventional_intensity": "Conventional SSL Intensity",
    "moral_intensity": "Moral SSL Intensity",
}
HEATMAP_OUTCOME_ORDER = [
    "any", "personal", "conventional", "moral",
    "total_intensity", "personal_intensity", "conventional_intensity", "moral_intensity",
]

# Edit these lists to control the correlation figures without changing the calculations above.
SPEARMAN_PLOT_OUTCOMES = [
    "total_intensity", "personal_intensity", "conventional_intensity", "moral_intensity",
]
HEATMAP_OUTCOMES = HEATMAP_OUTCOME_ORDER
HEATMAP_PREDICTORS = model_predictors
HEATMAP_ALPHA = 0.05
BOOTSTRAP_REPS = 1000
BOOTSTRAP_SEED = 20260530

regression_cols = model_predictors + list(ALL_OUTCOMES.values()) + [REPORT_WEIGHT_COL]
regression_df = df_analysis[regression_cols].apply(pd.to_numeric, errors="coerce").dropna().copy()
print(f"Regression/correlation complete-case dataframe: n={len(regression_df):,}; columns={regression_df.shape[1]:,}")

###############
# VIFs
###############
def compute_vif(data, predictors):
    X = data[predictors].astype(float).copy()
    variable_sds = X.std(axis=0)
    predictors_kept = variable_sds[variable_sds.gt(0)].index.tolist()
    X = sm.add_constant(X[predictors_kept], has_constant="add")

    rows = []
    for i, term in enumerate(X.columns):
        if term == "const":
            continue
        try:
            vif = variance_inflation_factor(X.to_numpy(), i)
        except Exception:
            vif = np.nan
        rows.append({
            "term": term,
            "predictor": label_pred(term) if "label_pred" in globals() else term,
            "vif": vif,
        })
    return pd.DataFrame(rows).sort_values("vif", ascending=False)

vif_table = compute_vif(regression_df, model_predictors)
display(vif_table)
vif_table.to_csv(OUTPUT_DIR / "regression_predictor_vifs.csv", index=False)

###############
# Zero-order Spearman correlations
###############
def effective_n(weights):
    weights = np.asarray(weights, dtype=float)
    weights = weights[np.isfinite(weights) & (weights > 0)]
    if len(weights) == 0:
        return np.nan
    return weights.sum() ** 2 / np.square(weights).sum()


def weighted_spearman_statsmodels(x, y, weights):
    """Weighted Spearman as DescrStatsW.corrcoef on average-rank transformed variables."""
    ranked = pd.DataFrame({
        "x_rank": stats.rankdata(x, method="average"),
        "y_rank": stats.rankdata(y, method="average"),
    })
    corr = DescrStatsW(ranked, weights=np.asarray(weights, dtype=float), ddof=0).corrcoef
    print(corr)
    return corr[0, 1]


def bootstrap_spearman_ci(pair, predictor, outcome_col, weight_col, rng, n_boot=BOOTSTRAP_REPS):
    """Percentile bootstrap CI for the weighted Spearman estimator."""
    n = len(pair)
    if n < 4:
        return np.nan, np.nan, np.nan

    boot = np.empty(n_boot, dtype=float)
    x = pair[predictor].to_numpy()
    y = pair[outcome_col].to_numpy()
    w = pair[weight_col].to_numpy()
    for b in range(n_boot):
        idx = rng.integers(0, n, size=n)
        try:
            boot[b] = weighted_spearman_statsmodels(x[idx], y[idx], w[idx])
        except Exception:
            boot[b] = np.nan

    boot = boot[np.isfinite(boot)]
    if len(boot) == 0:
        return np.nan, np.nan, np.nan
    return np.nanpercentile(boot, 2.5), np.nanpercentile(boot, 97.5), np.nanstd(boot, ddof=1)


def bootstrap_p_value(rho, boot_se):
    if pd.isna(rho) or pd.isna(boot_se) or boot_se <= 0:
        return np.nan
    z_stat = rho / boot_se
    return 2 * stats.norm.sf(abs(z_stat))


def spearman_rows(data, predictors, outcomes, weight_col=REPORT_WEIGHT_COL):
    rng = np.random.default_rng(BOOTSTRAP_SEED)
    rows = []
    for outcome_label, outcome_col in outcomes.items():
        for predictor in predictors:
            pair = data[[outcome_col, predictor, weight_col]].dropna()
            if len(pair) < 4 or pair[outcome_col].nunique() < 2 or pair[predictor].nunique() < 2:
                rho, ci_low, ci_high, boot_se, p_value, n_eff = np.nan, np.nan, np.nan, np.nan, np.nan, np.nan
            else:
                rho = weighted_spearman_statsmodels(pair[predictor], pair[outcome_col], pair[weight_col])
                n_eff = effective_n(pair[weight_col])
                ci_low, ci_high, boot_se = bootstrap_spearman_ci(pair, predictor, outcome_col, weight_col, rng)
                p_value = bootstrap_p_value(rho, boot_se)
            rows.append({
                "outcome": outcome_label,
                "outcome_label": OUTCOME_LABELS[outcome_label],
                "outcome_col": outcome_col,
                "term": predictor,
                "predictor": label_pred(predictor) if "label_pred" in globals() else predictor,
                "weight": weight_col,
                "spearman_r": rho,
                "ci_low": ci_low,
                "ci_high": ci_high,
                "bootstrap_se": boot_se,
                "p": p_value,
                "n": len(pair),
                "effective_n": n_eff,
                "bootstrap_reps": BOOTSTRAP_REPS,
            })
    return pd.DataFrame(rows)

CORRELATION_OUTCOMES = list(dict.fromkeys(SPEARMAN_PLOT_OUTCOMES + HEATMAP_OUTCOMES))
CORRELATION_PREDICTORS = list(dict.fromkeys(HEATMAP_PREDICTORS))
correlation_outcome_cols = {outcome: ALL_OUTCOMES[outcome] for outcome in CORRELATION_OUTCOMES}

spearman_table = spearman_rows(regression_df, CORRELATION_PREDICTORS, correlation_outcome_cols)
display(spearman_table.sort_values(["outcome", "spearman_r"], ascending=[True, False]))
spearman_table.to_csv(OUTPUT_DIR / "regression_zero_order_spearman.csv", index=False)

###############
# Correlation plots
###############
def plot_zero_order_correlations(corr_df, outcome_label):
    pal = make_aesthetic(font_scale=0.9)
    plot_df = corr_df[corr_df["outcome"].eq(outcome_label)].copy()
    plot_df = plot_df.sort_values("spearman_r")
    colors = np.where(plot_df["p"].lt(0.05), pal[0], "#aaaaaa")

    fig, ax = plt.subplots(figsize=(9, 8))
    for color, (_, row) in zip(colors, plot_df.iterrows()):
        ax.errorbar(
            row["spearman_r"], row["predictor"],
            xerr=[[row["spearman_r"] - row["ci_low"]], [row["ci_high"] - row["spearman_r"]]],
            fmt="o", color=color, markersize=4.5, capsize=2.5, linewidth=0.9, elinewidth=0.8, zorder=3,
        )
    ax.axvline(0, color="#cccccc", linewidth=0.8)
    ax.set_title(OUTCOME_LABELS[outcome_label])
    ax.set_xlabel("Weighted zero-order Spearman r  (95% CI)")
    ax.set_ylabel("")
    ax.set_xlim(-1, 1)
    sns.despine(ax=ax)
    fig.subplots_adjust(left=0.42, right=0.98, bottom=0.10, top=0.92)

    out_path = OUTPUT_DIR / f"zero_order_spearman_{outcome_label}.pdf"
    fig.savefig(out_path, bbox_inches="tight")
    plt.show()
    return out_path

spearman_plot_paths = [plot_zero_order_correlations(spearman_table, outcome) for outcome in SPEARMAN_PLOT_OUTCOMES]

heatmap_cols = [OUTCOME_LABELS[outcome] for outcome in HEATMAP_OUTCOMES]
heatmap_predictors = [label_pred(term) if "label_pred" in globals() else term for term in HEATMAP_PREDICTORS]
heatmap_df = spearman_table.pivot(index="predictor", columns="outcome_label", values="spearman_r")
heatmap_p = spearman_table.pivot(index="predictor", columns="outcome_label", values="p")
heatmap_df = heatmap_df.reindex(index=heatmap_predictors, columns=heatmap_cols)
heatmap_p = heatmap_p.reindex(index=heatmap_predictors, columns=heatmap_cols)
heatmap_sig = heatmap_p.lt(HEATMAP_ALPHA)
heatmap_plot_df = heatmap_df.where(heatmap_sig)

heatmap_cmap = sns.color_palette("vlag", as_cmap=True)
heatmap_cmap.set_bad("#e6e6e6")

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    heatmap_plot_df,
    ax=ax,
    cmap=heatmap_cmap,
    center=0,
    vmin=-1,
    vmax=1,
    linewidths=0.25,
    linecolor="white",
    cbar_kws={"label": "Weighted Spearman r"},
)
ax.set_title(f"Weighted Zero-order Spearman Correlations (gray = p ≥ {HEATMAP_ALPHA:.2f})")
ax.set_xlabel("")
ax.set_ylabel("")
fig.subplots_adjust(left=0.36, right=0.98, bottom=0.24, top=0.92)
heatmap_path = OUTPUT_DIR / "zero_order_spearman_heatmap.pdf"
fig.savefig(heatmap_path, bbox_inches="tight")
plt.show()

print("Saved VIF and bootstrap-CI weighted zero-order Spearman outputs:")
print("-", OUTPUT_DIR / "regression_predictor_vifs.csv")
print("-", OUTPUT_DIR / "regression_zero_order_spearman.csv")
for path in spearman_plot_paths + [heatmap_path]:
    print("-", path)


In [ ]:
####################
# Install CmdStanPy
####################

!pip -q install cmdstanpy
!python -m cmdstanpy.install_cmdstan --quiet


####################
# Imports
####################

import json
import numpy as np
import cmdstanpy as csp


####################
# Data
####################

N_PER_ESTIMATE = 800

p1_mean = np.array([0.56, 0.50, 0.45])
p2_mean = np.array([0.20, 0.24, 0.18, 0.22])


####################
# Helpers
####################

def logit(p):
    """Convert probabilities to log odds."""
    return np.log(p / (1 - p))


def logit_se(p, n):
    """Approximate the standard error of log odds."""
    return np.sqrt(1 / (n * p) + 1 / (n * (1 - p)))


def make_stan_data(p1_mean, p2_mean, n_per_estimate):
    """Create Stan data on the log-odds scale."""
    return {
        "N1": len(p1_mean),
        "p1_logodds": logit(p1_mean).tolist(),
        "p1_logodds_se": logit_se(p1_mean, n_per_estimate).tolist(),
        "N2": len(p2_mean),
        "p2_logodds": logit(p2_mean).tolist(),
        "p2_logodds_se": logit_se(p2_mean, n_per_estimate).tolist(),
    }


####################
# Stan Model
####################

stan_code = """
data {
  int<lower=0> N1;
  int<lower=0> N2;

  vector[N1] p1_logodds;
  vector<lower=0>[N1] p1_logodds_se;

  vector[N2] p2_logodds;
  vector<lower=0>[N2] p2_logodds_se;
}

parameters {
  real alpha1;
  real alpha2;
}

transformed parameters {
  real<lower=0, upper=1> theta1 = inv_logit(alpha1);
  real<lower=0, upper=1> theta2 = inv_logit(alpha2);
}

model {
  p1_logodds ~ normal(alpha1, p1_logodds_se);
  p2_logodds ~ normal(alpha2, p2_logodds_se);

  theta1 ~ beta(2, 2);
  theta2 ~ beta(2, 2);
}

generated quantities {
  real<lower=0, upper=1> p_friend = theta1 * theta2;
}
"""

with open("friends_logodds.stan", "w") as f:
    f.write(stan_code)


####################
# Save Data
####################

stan_data = make_stan_data(
    p1_mean=p1_mean,
    p2_mean=p2_mean,
    n_per_estimate=N_PER_ESTIMATE,
)

with open("data_logodds.json", "w") as f:
    json.dump(stan_data, f, indent=4)

print(json.dumps(stan_data, indent=4))


####################
# Fit Model
####################

model = csp.CmdStanModel(stan_file="friends_logodds.stan")
fit = model.sample(
    data="data_logodds.json",
    chains=4,
    iter_warmup=1000,
    iter_sampling=1000,
    seed=123,
)

####################
# Posterior Summary
####################

summary = fit.summary(sig_figs=3)
display(summary.loc[["theta1", "theta2", "p_friend"]])


####################
# 90% Posterior Intervals
####################

draws = fit.draws_pd()
for name in ["theta1", "theta2", "p_friend"]:
    mean = draws[name].mean()
    lower = draws[name].quantile(0.05)
    upper = draws[name].quantile(0.95)

    print(f"{name}: mean = {mean:.3f}, 90% posterior interval = ({lower:.3f}, {upper:.3f})")

In [ ]:
# regression_run

###############
# Predictors and outcomes
###############
model_predictors = [
    "age_num_z", "gender_male", "gender_other", "race_white", "race_black",
    "pid_republican", "pid_democrat", "income_ord_z", "college_degree",
    "employed", "retired", "ideo_num_z",
    "use_freq_code_z", "aias4_score_z", "anthrotech_score_z",
    "sias4_score_z", "lsns6_score_z",
    "tipi_extraversion_z", "tipi_agreeableness_z", "tipi_conscientiousness_z",
    "tipi_emotional_stability_z", "tipi_openness_z", "ai_social_use_index_z",
]

LOGIT_OUTCOMES = {
    "any":          "used_any_ssl",
    "personal":     "used_personal_ssl",
    "conventional": "used_conventional_ssl",
    "moral":        "used_moral_ssl",
}

OLS_OUTCOMES = {
    "total_intensity":        "ssl_intensity_total",
    "personal_intensity":     "personal_ssl_intensity",
    "conventional_intensity": "societal_conventional_ssl_intensity",
    "moral_intensity":        "moral_ssl_intensity",
}

###############
# Fit all models
###############
logit_models_uw  = {}   # {label: statsmodels result}
logit_results    = []
for lbl, col in LOGIT_OUTCOMES.items():
    model, tbl = fit_logit(col, lbl)
    logit_models_uw[lbl] = model
    logit_results.append(tbl)
    tbl_wt = fit_svy_logit(col, lbl)
    logit_results.append(tbl_wt)
    print(f"Logit {lbl}: n={tbl['n'].iloc[0]}")

ols_models_uw  = {}
ols_results    = []
for lbl, col in OLS_OUTCOMES.items():
    model, tbl = fit_ols(col, lbl)
    ols_models_uw[lbl] = model
    ols_results.append(tbl)
    tbl_wt = fit_svy_ols(col, lbl)
    ols_results.append(tbl_wt)
    print(f"OLS {lbl}: n={tbl['n'].iloc[0]}")

all_logit = pd.concat(logit_results, ignore_index=True)
all_ols   = pd.concat(ols_results,   ignore_index=True)


In [ ]:
# regression_plot_and_tables

###############
# Plot — one figure per outcome, weighted + unweighted overlaid
###############
make_aesthetic()

def _frame_for(results_df, outcome, model_label):
    """Extract a clean DataFrame for MultiModelHandler from the results table."""
    df = results_df[(results_df["outcome"].eq(outcome)) &
                    (results_df["model"].eq(model_label))].copy()
    df["term"] = df["term"].replace("_intercept_","const")
    return df[["term","coef","ci_low","ci_high","p"]]

for lbl in LOGIT_OUTCOMES:
    handler = MultiModelHandler({
        "Unweighted": logit_models_uw[lbl],
        "Weighted":   _frame_for(all_logit, lbl, "weighted"),
    })
    fig, ax = handler.plot(exp=True, figsize=(9, 8))
    ax.set_xlabel("Odds ratio  (95% CI)")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"logit_or_{lbl}.pdf", bbox_inches="tight")
    plt.show()

for lbl in OLS_OUTCOMES:
    handler = MultiModelHandler({
        "Unweighted": ols_models_uw[lbl],
        "Weighted":   _frame_for(all_ols, lbl, "weighted"),
    })
    fig, ax = handler.plot(figsize=(9, 8))
    ax.set_xlabel("Coefficient  (95% CI)")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"ols_coef_{lbl}.pdf", bbox_inches="tight")
    plt.show()


###############
# Tables — CSV + LaTeX
###############
all_logit.to_csv(OUTPUT_DIR / "regression_logit.csv", index=False)
all_ols.to_csv(OUTPUT_DIR   / "regression_ols.csv",   index=False)

for name, df in [("regression_logit", all_logit), ("regression_ols", all_ols)]:
    clean = (df[df["term"].ne("const")]
             .assign(p_fmt=lambda d: d["p"].apply(
                 lambda p: "<.001" if p<.001 else f"{p:.3f}"))
             .round({"coef":3,"ci_low":3,"ci_high":3}))
    caption = ("Logistic regression coefficients (log-odds) predicting SSL domain use."
               if "logit" in name else
               "OLS regression coefficients predicting SSL use intensity.")
    print(clean.to_latex(index=False, caption=caption, label=f"tab:{name}",
                         columns=["outcome","model","term","coef","ci_low","ci_high","p_fmt"]))
    with open(OUTPUT_DIR / f"{name}.tex", "w") as f:
        f.write(clean.to_latex(index=False, caption=caption, label=f"tab:{name}",
                               columns=["outcome","model","term","coef","ci_low","ci_high","p_fmt"]))
print("Saved tables.")
